# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
import os
warnings.filterwarnings("ignore")

# sklearn utilities
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted, validate_data
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils import check_random_state
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import QuantileRegressor
from scipy.stats import norm, t, logistic, cauchy, gumbel_r, pearsonr, spearmanr
from itertools import product

# LP-based quantile regression
from sklearn.linear_model import QuantileRegressor

#mse
from sklearn.metrics import mean_squared_error

# Statsmodels (IRLS quantile regression)
import statsmodels.api as sm

from scipy.stats import norm
from scipy.stats import linregress
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, LogFormatterMathtext

# SGD Quantile Regression Class

In [ ]:
class SGDQuantileRegressor(RegressorMixin, BaseEstimator):
    '''
    Linear quantile regression estimator trained via proximal stochastic gradient descent.

    This estimator fits a linear model of the form
        y ≈ X beta + b
    by minimizing a regularized quantile regression objective based on the pinball loss:

        min_{beta, b}
            (1/n) * sum_{i=1}^n pinball_loss(y_i - x_i^T beta - b)
            + l1 * ||beta||_1 + (l2/2) * ||beta||_2^2,

    The intercept b is left unpenalized, while the coefficient vector beta is regularized via l1 (l2 regularization
    with a small default coefficient of 10^{-6} is included for numerical stability).

    Optimization is performed using proximal stochastic subgradient descent with
    mini-batches. At each iteration, a stochastic subgradient of the pinball loss is
    computed, followed by a proximal update that applies soft-thresholding (L1) and
    ridge shrinkage (L2) to the coefficients. Two step-size regimes are supported:
    square-root decay and AdaGrad with per-coordinate adaptive learning rates.
    Computations are performed in a user-specified floating-point precision (dtype),
    allowing a trade-off between numerical stability and memory efficiency.

    Optionally, Polyak/Ruppert iterate averaging is used to stabilize optimization and
    reduce variance, particularly in the presence of stochastic updates and nonsmooth
    loss functions. The final model parameters are taken as either the averaged
    iterates or the last raw iterate, depending on the `use_averaging` setting.

    For compatibility with scikit-learn, sample weights are supported via exact sample
    replication for nonnegative integer weights, ensuring that the weighted objective
    is equivalent to a replicated empirical risk minimization problem.

    Parameters
    ----------
    quantile : float, default=0.5
        Target quantile level tau in (0, 1). For example, tau=0.5 corresponds to median
        regression.

    max_iter : int, default=500
        Maximum number of proximal SGD iterations.

    base_lr : float, default=0.2
        Base learning rate used in the step-size schedule.

    batch_size : int, default=32
        Mini-batch size used for stochastic gradient updates.

    l1 : float, default=0.0
        L1 regularization strength (lasso penalty) applied to beta.

    l2 : float, default=0.0
        L2 regularization strength (ridge penalty) applied to beta (can add numerical stability).

    eval_every : int, default=10
        Frequency (in iterations) at which training loss is evaluated and recorded.

    random_state : int or None, default=None
        Seed for the random number generator used in mini-batch sampling.

    verbose : bool, default=False
        If True, print training diagnostics during optimization.

    use_adagrad : bool, default=False
        If True, use AdaGrad adaptive step sizes instead of square-root decay.

    adagrad_eps : float, default=1e-8
        Numerical stabilizer used in AdaGrad step-size computation.

    use_averaging : bool, default=True
        If True, use Polyak/Ruppert averaging of iterates; otherwise, return the last
        raw proximal SGD iterate.

    dtype : numpy.dtype, default=numpy.float64
        Floating-point precision used for internal computations and model parameters.
        Using float32 may reduce memory usage and improve speed, but can reduce numerical
        stability, especially for nonsmooth optimization and small regularization values.

    early_stopping : bool, default=False
        If True, hold out a validation set and apply early stopping based on validation
        pinball loss.

    validation_fraction : float, default=0.1
        Fraction of the training data to set aside as validation when early_stopping=True.

    tol : float, default=1e-4
        Minimum decrease in validation loss required to qualify as an improvement for
        early stopping.

    n_iter_no_change : int, default=10
        Stop training if validation loss does not improve by at least tol for this many
        consecutive evaluation checks (evaluations occur every eval_every iterations).

    restore_best_weights : bool, default=True
        If True and early_stopping=True, restore the model parameters corresponding to
        the best validation loss before returning.

    Attributes
    ----------
    coef_ : ndarray of shape (n_features,)
        Estimated coefficient vector beta.
    intercept_ : float
        Estimated intercept b.
    n_iter_ : int
        Number of iterations performed.
    history_ : dict
        Training diagnostics recorded during optimization.
        Contains:
        - 'iter': list of iteration indices at which evaluation occurred,
        - 'train_pinball_loss': list of mean training pinball losses at those iterations,
        - 'val_pinball_loss': list of mean validation pinball losses (only recorded when
            early_stopping=True; otherwise this list remains empty).

    best_val_loss_ : float or None
        Best (lowest) validation pinball loss observed during training when early stopping
        is enabled. Set to None if early_stopping=False.

    best_train_loss_ : float or None
        Training pinball loss at the iteration where best_val_loss_ was achieved
        (only recorded when early_stopping=True).

    best_iter_ : int or None
        Iteration index at which best_val_loss_ was achieved when early stopping is enabled.
        Set to None if early_stopping=False.

    Notes
    -----
    This estimator implements a proximal stochastic optimization scheme suitable for
    convex and nonsmooth objectives. It is designed to be compatible with scikit-learn's
    estimator API, including input validation, reproducibility via random_state, and
    sklearn-style sample weighting.
    '''

    def __init__(self, quantile=0.5, max_iter=1000, base_lr=0.5, batch_size=256,
                 l1=0.0, l2=0.0, eval_every=10, random_state=None, verbose=False,
                 use_adagrad=False, adagrad_eps=1e-8, use_averaging=True, dtype=np.float64,
                 early_stopping=False, validation_fraction=0.1, tol=1e-4, n_iter_no_change=10,
                 restore_best_weights=True):
        self.quantile = quantile
        self.max_iter = max_iter
        self.base_lr = base_lr
        self.batch_size = batch_size
        self.l1 = l1
        self.l2 = l2
        self.eval_every = eval_every
        self.random_state = random_state
        self.verbose = verbose
        self.use_adagrad = use_adagrad
        self.adagrad_eps = adagrad_eps
        self.use_averaging = use_averaging
        self.dtype=dtype
        self.early_stopping = early_stopping
        self.validation_fraction = validation_fraction
        self.tol = tol
        self.n_iter_no_change = n_iter_no_change
        self.restore_best_weights = restore_best_weights

    def _train_val_split(self, X, y, rng, validation_fraction):
        """Split X,y into train/val subsets for early stopping."""
        if not (0.0 < validation_fraction < 1.0):
            raise ValueError("validation_fraction must be in (0, 1).")
        n = X.shape[0]
        n_val = max(1, int(np.floor(validation_fraction * n)))
        if n_val >= n:
            raise ValueError("validation_fraction is too large; validation set would be empty train set.")

        perm = rng.permutation(n)
        val_idx = perm[:n_val]
        train_idx = perm[n_val:]

        return X[train_idx], y[train_idx], X[val_idx], y[val_idx]

    def _get_effective_params(self, beta, intercept, beta_avg, intercept_avg):
        """Return the parameters that are currently considered the model parameters."""
        if self.use_averaging:
            return beta_avg, intercept_avg
        return beta, intercept

    def _copy_params(self, beta, intercept):
        """Deep-copy params for snapshotting best model."""
        return beta.copy(), self.dtype(intercept)

    def _is_improvement(self, current, best):
        """Return True if current is better (lower) than best by at least tol."""
        return current < (best - self.tol)

    def _early_stopping_step(self, loss_value, best_loss, no_improve_count,
                             best_beta, best_intercept, current_beta,
                             current_intercept):
        """
        One update step for early stopping bookkeeping.
        """
        if best_loss is None or self._is_improvement(loss_value, best_loss):
            best_loss = loss_value
            no_improve_count = 0
            best_beta, best_intercept = self._copy_params(current_beta, current_intercept)
            should_stop = False
        else:
            no_improve_count += 1
            should_stop = (no_improve_count >= self.n_iter_no_change)

        return best_loss, no_improve_count, best_beta, best_intercept, should_stop


    def _sqrt_decay_lr(self, base_lr, t):
        '''
        Helper method for .fit(). Square-root learning-rate decay schedule.

        Returns the step size
             base_lr / sqrt(t)
        at iterate t. This is commonly used for stochastic (sub)gradient methods in convex / nonsmooth settings.
        The 1/sqrt(t) decay reduces gradient noise over time and is compatible with
        standard convergence guarantees when using averaged iterates.

        Parameters
        ----------
        base_lr : float
            Initial step size η_0.
        t : int
            Iteration counter (starts at 1 to avoid division by zero).

        Returns
        -------
        step_size : float
            Learning rate at iteration t.
        '''

        return base_lr / np.sqrt(t)

    def _update_running_average(self, avg, current, t):
        '''
        Update a running average of an iterate sequence.
        Since this applies to both the model coeficcient beta and intercept term b,
        we will let theta denote an arbitrary parameter(s).

        Given the previous running average avg = \\bar{theta}_{t-1} and the current iterate
        current = theta_t, this returns
            \\bar{theta}_t = \\bar{theta}_{t-1} + (theta_t - \\bar{theta}_{t-1}) / t,
        which is exactly the arithmetic mean of {\theta_1, ..., theta_t}.

        This type of iterate averaging (Polyak/Ruppert averaging) is frequently used with SGD /
        stochastic subgradient methods to reduce variance and improve stability, especially
        for convex nonsmooth objectives.

        Parameters
        ----------
        avg : float or ndarray
            Previous running average \\bar{θ}_{t-1}.
        current : float or ndarray
            Current iterate theta_t.
        t : int
            Iteration counter (t >= 1).

        Returns
        -------
        avg_new : float or ndarray
            Updated running average \\bar{θ}_t.
        '''

        return avg + (current - avg) / t

    def _pinball_loss_and_subgrad(self, y, y_pred):
        '''
        Helper method used in .fit(). Computes the mean pinball (check) loss for quantile
        tau and a subgradient w.r.t. residuals r = y-y_pred. For residual r_i = y_i - y_pred_i,
        the pinball loss is loss(r_i) = tau * max(r_i, 0) + (1-tau) * max(-r_i, 0).

        For a given quantile level tau, the pinball loss for residual r_i = y_i - y_pred_i is
            pinball_loss(r_i) = tau max(r_i, 0) + (1 - tau) max(-r_i, 0).

        This method returns:
        (i) the mean pinball loss over the samples, and
        (ii) a vector g of subgradients of the pinball loss with respect to the residuals r,
            where each component is chosen as
                g_i = tau - 1{r_i < 0}.
        At the nondifferentiable point r_i = 0, this corresponds to selecting the
        subgradient g_i = tau, which lies in the valid subdifferential interval [tau - 1, tau].

        Note that g represents subgradients with respect to residuals r = y - y_pred.
        Gradients with respect to model parameters are obtained later via the chain rule,
        which introduces an additional minus sign.

        Parameters
        ----------
        y : ndarray of shape (m,)
            True target values for the mini-batch.
        y_pred : ndarray of shape (m,)
            Predicted values for the mini-batch.

        Returns
        -------
        loss : float
            Mean pinball loss over the mini-batch.
        g : ndarray of shape (m,)
            Subgradient of the pinball loss with respect to the residuals r = y - y_pred.
        '''
        r = y - y_pred
        g = self.quantile - (r < 0).astype(float)
        per_sample = (
            self.quantile * np.maximum(r, 0.0)
            + (1.0 - self.quantile) * np.maximum(-r, 0.0)
        )
        loss = float(np.mean(per_sample))
        return loss, g

    def _batch_gradient(self, X, y, beta, intercept):
        '''
        Helper method used in .fit(). Computes the mini-batch subgradient of the mean pinball loss
        w.r.t. (beta, intercept). Our mini-batch objective is:
            L(beta, b) = (1/m) * sum_{i=1}^m pinball_loss(r_i),
        where r_i = y_i - y_pred_i and y_pred_i = x_i^T beta + b.

        In this implementation, g is computed by `_pinball_loss_and_subgrad` as
            g_i = tau - 1{r_i < 0},
        which selects g_i = tau at the nondifferentiable point r_i = 0. See _pinball_loss_and_subgrad
        documentation for more detail.

        By the chain rule, since r_i = y_i - x_i^T beta - b, we have the parial of r_i w.r.t. beta
            = -x_i
        and the partial of r_i w.r.t. the intercept b
         = -1.
        So the subgradients of the mean loss with respect to the parameters are
            ∇_beta L = -(1/m) * X^T g,
            ∇_b    L = -(1/m) * 1^T g = -mean(g),

        where X \\in R^{m×d} is the mini-batch design matrix and g \\in R^m stacks the per-sample
        residual subgradients.

        Notes
        -----
        - This method returns gradients for the *data-fit* term only (pinball loss). L1/L2
        regularization is handled separately via the proximal operator in the update step.
        - The negative signs appear because residuals are defined as r = y - y_pred.

        Parameters
        ----------
        X : ndarray of shape (m, d)
            Mini-batch feature matrix.
        y : ndarray of shape (m,)
            Mini-batch targets.
        beta : ndarray of shape (d,)
            Current coefficient vector.
        intercept : float
            Current intercept.

        Returns
        -------
        grad_beta : ndarray of shape (d,)
            Mini-batch (sub)gradient of the mean pinball loss with respect to beta.
        grad_intercept : float
            Mini-batch (sub)gradient of the mean pinball loss with respect to the intercept.
        '''
        y_pred = X @ beta + intercept
        _, g = self._pinball_loss_and_subgrad(y, y_pred)
        grad_beta = -(X.T @ g) / X.shape[0]
        grad_intercept = -g.mean()

        return grad_beta, grad_intercept

    def _prox_elastic_net(self, beta, l1, l2, step_size):
        '''
        Helper method for .fit(). Applies the proximal operator of the elastic-net penalty to the vector beta.

        This method computes the proximal map of the function
            g(beta) = l1 * ||beta||_1 + (l2/2) * ||beta||_2^2,
        evaluated at the point beta with step_size. That is, it returns

            prox_{step_size, g}(beta)
            = argmin_z { (1/2)||z - beta||_2^2 + step_size * l1 ||z||_1 + step_size * (l2/2)||z||_2^2 }.

        The elastic-net proximal operator is separable across coordinates and admits the
        closed-form expression
            prox_{step_size, g}(beta) = (1 / (1 + step_size * l2)) * S_{step_size, l1}(beta),

        where S_{k}(·) is the soft-thresholding operator defined componentwise by
            S_{k}(beta_j) = sign(beta_j) * max(|beta_j| - k, 0).

        Parameters
        ----------
        beta : ndarray of shape (d,)
            Input vector to be prox-mapped (typically the post-gradient iterate).
        l1 : float
            L1 regularization strength (promotes sparsity).
        l2 : float
            L2 regularization strength (ridge shrinkage). Used for stability.
        step_size : float or ndarray of shape (d,)
            Step size. If an array is provided (e.g., in AdaGrad), the proximal map is
            applied coordinate-wise using per-feature step sizes.

        Returns
        -------
        beta_prox : ndarray of shape (d,)
            Coefficient vector after applying the elastic-net proximal operator.
        '''

        # step_size can be float or shape (n_features,)
        thresh = step_size * l1
        beta_soft = np.sign(beta) * np.maximum(np.abs(beta) - thresh, 0.0)

        return beta_soft / (1.0 + step_size * l2)
        # old implementation
        # beta = beta / (1.0 + step_size * l2)
        # thresh = step_size * l1
        # return np.sign(beta) * np.maximum(np.abs(beta) - thresh, 0.0)

    def _sgd_prox_step(self, X, y, beta, intercept, step_size, *, l1=0.0, l2=0.0):
        '''
        Helper method for .fit(). Performs one proximal SGD (proximal stochastic subgradient) update on a mini-batch.

        We minimize an objective of the form
            f(beta, b) + g(beta),
        where f is the mean pinball loss (quantile regression data-fit term) and
            g(beta) = l1 * ||beta||_1 + (l2/2) * ||beta||_2^2
        is an elastic-net penalty applied to the coefficients only (the intercept is unpenalized).
        Primary purpose of the l2 penalty is numerical stability.

        Given a mini-batch (X, y), this method computes a stochastic subgradient of f and applies:
            v_beta = beta - step_size * \\hat{grad}_beta f(beta, b)   (forward / gradient step)
            b_next = b - step_size * \\hat{grad}_b    f(beta, b)   (SGD step on intercept)
            beta_next = prox_{step_size, g}(v_beta)    (backward / proximal step)

        The proximal operator for the elastic-net penalty is applied coordinate-wise via
        `_prox_elastic_net`, which combines soft-thresholding (L1) and ridge shrinkage (L2).

        Parameters
        ----------
        X : ndarray of shape (m, d)
            Mini-batch feature matrix.
        y : ndarray of shape (m,)
            Mini-batch targets.
        beta : ndarray of shape (d,)
            Current coefficient vector.
        intercept : float
            Current intercept.
        step_size : float or ndarray of shape (d,)
            Step size. Typically a scalar for standard SGD, or a vector of per-feature
            step sizes when using AdaGrad.
        l1 : float, default=0.0
            L1 regularization strength.
        l2 : float, default=0.0
            L2 regularization strength.

        Returns
        -------
        beta_next : ndarray of shape (d,)
            Updated coefficient vector after the gradient and proximal steps.
        intercept_next : float
            Updated intercept after the SGD step.
        '''
        grad_beta, grad_intercept = self._batch_gradient(X, y, beta, intercept)

        beta = beta - step_size * grad_beta
        intercept = intercept - step_size * grad_intercept

        if l1 > 0 or l2 > 0:
            beta = self._prox_elastic_net(beta, l1, l2, step_size)

        return beta, intercept

    def _init_model_params(self, n_features, y):
        '''
        Helper method for .fit(). Initializes model parameters (beta, intercept) for quantile
        regression training.

        We initialize the coefficient vector to zero and choose the intercept as an
        empirical tau-quantile of the targets. With beta = 0, the model predicts a constant
        value b, and the minimizer of the empirical pinball loss
            (1/n) * sum_i pinball_loss(y_i - b)
        over b is any empirical tau-quantile of {y_i}. Thus, setting
            intercept_0 = quantile_tau(y)
        provides a principled "best constant predictor" initialization for the pinball loss.

        Parameters
        ----------
        n_features : int
            Number of features (dimension of beta).
        y : ndarray of shape (n,)
            Target values (used to initialize the intercept via an empirical quantile).

        Returns
        -------
        beta : ndarray of shape (n_features,)
            Initial coefficient vector (all zeros).
        intercept : float
            Initial intercept, chosen as the empirical τ-quantile of y.
        '''
        beta = np.zeros(n_features, dtype=self.dtype)
        intercept = self.dtype(np.quantile(y, self.quantile))

        return beta, intercept

    def _pinball_loss_from_params(self, X, y_true, beta, intercept):
        '''
        Helper method for .fit(). Computes the mean pinball (check) loss for given model parameters.

        This method evaluates the quantile regression objective for fixed parameters
        (beta, intercept) on a dataset (X, y_true). For quantile level tau and residuals
            r_i = y_i - (x_i^T beta + intercept),
        the pinball loss is defined as
            pinball_loss(r_i) = tau * max(r_i, 0) + (1 - tau) * max(-r_i, 0).

        The returned value is the empirical mean of the pinball loss:
            (1/n) * sum_{i=1}^n pinball_loss(r_i).

        Unlike `_pinball_loss_and_subgrad`, this method does not compute subgradients and
        is used purely for objective evaluation, e.g., in diagnostics and model assessment.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Feature matrix.
        y_true : ndarray of shape (n_samples,)
            True target values.
        beta : ndarray of shape (n_features,)
            Coefficient vector at which to evaluate the loss.
        intercept : float
            Intercept term at which to evaluate the loss.

        Returns
        -------
        loss : float
            Mean pinball loss evaluated at (beta, intercept).
        '''
        y_true = np.asarray(y_true, dtype=float).reshape(-1)
        y_pred = X @ beta + intercept
        r = y_true - y_pred
        per_sample = (
            self.quantile * np.maximum(r, 0.0)+ (1.0 - self.quantile) * np.maximum(-r, 0.0)
            )

        return float(np.mean(per_sample))

    def _kkt_stationarity_vector(self, X, y, beta, intercept):
        """
        Compute the unregularized stationarity proxy

            g = (1/n) X^T s,

        where s_i = tau - 1{r_i < 0} and
        r_i = y_i - (x_i^T beta + intercept).

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
        y : ndarray of shape (n_samples,)
        beta : ndarray of shape (n_features,)
        intercept : float

        Returns
        -------
        g : ndarray of shape (n_features,)
            Stationarity proxy vector for the empirical pinball loss.
        """
        X = np.asarray(X, dtype=self.dtype)
        y = np.asarray(y, dtype=self.dtype).reshape(-1)
        beta = np.asarray(beta, dtype=self.dtype).reshape(-1)
        intercept = self.dtype(intercept)

        r = y - (X @ beta + intercept)
        s = self.quantile - (r < 0).astype(self.dtype)
        g = (X.T @ s) / X.shape[0]

        return np.asarray(g, dtype=self.dtype)

    def _kkt_grad_inf(self, X, y, beta, intercept):
        """
        Compute ||g||_inf for the unregularized stationarity proxy

            g = (1/n) X^T s.
        """
        g = self._kkt_stationarity_vector(X, y, beta, intercept)
        return float(np.max(np.abs(g)))

    def fit(
        self,
        X,
        y,
        *,
        sample_weight=None,
        X_monitor=None,
        y_monitor=None,
        monitor_name="monitor",
        record_monitor_loss=True,
        record_train_kkt=False,
        record_monitor_kkt=False,
        store_param_path=False,
    ):
        '''
        Fits a linear quantile regression model using proximal stochastic gradient descent.

        This method solves the regularized quantile regression problem

            min_{beta, b}
                (1/n) * sum_{i=1}^n pinball_loss(y_i - x_i^T beta - b)
                + l1 * ||beta||_1 + (l2/2) * ||beta||_2^2,

        where pinball_loss is the pinball (check) loss at quantile level tau, beta is the coefficient
        vector, and b is the intercept. The intercept is unpenalized, while the coefficients
        are regularized via an l1 penalty (l2 term is included for numerical stability with default
        coefficient 10^{-6} so l1 regularization solution is minimally impacted).

        Optimization is performed using proximal stochastic subgradient descent with
        mini-batches. At each iteration:

        1) A mini-batch is sampled from the training data.
        2) A stochastic subgradient of the pinball loss is computed.
        3) A gradient step is taken on (beta, b).
        4) The elastic-net proximal operator is applied to beta.
        5) Running averages of beta and b are updated (Polyak/Ruppert averaging, if use_averaging=True).

        Two step-size regimes are supported:

        - Square-root decay SGD: step_size_t = base_lr / sqrt(t).
        - AdaGrad: per-coordinate adaptive step sizes based on accumulated squared gradients.

        All computations are carried out in the precision specified by `dtype`.

        By default, the final model parameters are taken as Polyak/Ruppert averages of the
        iterates rather than the last raw iterate, which typically improves stability for
        stochastic and nonsmooth optimization. If `use_averaging=False`, the last raw
        iterate is returned instead.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Training feature matrix.
        y : ndarray of shape (n_samples,)
            Target values.
        sample_weight : ndarray of shape (n_samples,), optional
            Sample weights. For sklearn-compatible behavior, integer-valued weights are
            implemented via explicit data replication.
        X_monitor : ndarray of shape (n_monitor, n_features), optional
            External monitoring feature matrix. This can be a held-out test set used
            to record convergence diagnostics during training.
        y_monitor : ndarray of shape (n_monitor,), optional
            Targets corresponding to X_monitor.
        monitor_name : str, default="monitor"
            Descriptive label for the external monitoring set. Currently cosmetic.
        record_monitor_loss : bool, default=True
            If True and X_monitor/y_monitor are provided, record monitor pinball loss.
        record_train_kkt : bool, default=False
            If True, record the training-set KKT proxy ||g||_inf at evaluation times.
        record_monitor_kkt : bool, default=False
            If True and X_monitor/y_monitor are provided, record the monitor-set KKT proxy
            ||g||_inf at evaluation times.
        store_param_path : bool, default=False
            If True, store the effective parameter path at evaluation times.

        Returns
        -------
        self : object
            Fitted estimator. The learned parameters are stored in `coef_` and `intercept_`,
            and training diagnostics are available in `history_`.
        '''

        if not (0.0 < self.quantile < 1.0):
            raise ValueError('quantile must be in (0, 1)')
        if not isinstance(self.batch_size, (int, np.integer)) or self.batch_size <= 0:
            raise ValueError('batch_size must be a positive integer.')
        if not isinstance(self.max_iter, (int, np.integer)) or self.max_iter <= 0:
            raise ValueError('max_iter must be a positive integer.')
        if not isinstance(self.eval_every, (int, np.integer)) or self.eval_every <= 0:
            raise ValueError('eval_every must be a positive integer.')
        if self.early_stopping:
            if not (0.0 < self.validation_fraction < 1.0):
                raise ValueError("validation_fraction must be in (0, 1).")
            if not isinstance(self.n_iter_no_change, (int, np.integer)) or self.n_iter_no_change <= 0:
                raise ValueError("n_iter_no_change must be a positive integer.")
            if self.tol < 0:
                raise ValueError("tol must be nonnegative.")

        # --- sklearn input validation ---
        X, y = validate_data(self, X, y, accept_sparse=False, y_numeric=True, reset=True)
        X = np.asarray(X, dtype=self.dtype)
        y = np.asarray(y, dtype=self.dtype).reshape(-1)

        # --- optional external monitor data validation ---
        if (X_monitor is None) != (y_monitor is None):
            raise ValueError("X_monitor and y_monitor must either both be provided or both be None.")

        if X_monitor is not None:
            X_monitor = np.asarray(X_monitor, dtype=self.dtype)
            y_monitor = np.asarray(y_monitor, dtype=self.dtype).reshape(-1)
            if X_monitor.ndim != 2:
                raise ValueError("X_monitor must be a 2D array.")
            if y_monitor.ndim != 1:
                raise ValueError("y_monitor must be a 1D array.")
            if X_monitor.shape[0] != y_monitor.shape[0]:
                raise ValueError(
                    f"X_monitor and y_monitor must have the same number of rows; "
                    f"got {X_monitor.shape[0]} and {y_monitor.shape[0]}."
                )
            if X_monitor.shape[1] != X.shape[1]:
                raise ValueError(
                    f"X_monitor must have {X.shape[1]} features, got {X_monitor.shape[1]}."
                )

        # --- sklearn-compatible sample_weight: expand to repeated/removed rows ---
        if sample_weight is not None:
            sw = np.asarray(sample_weight, dtype=float).reshape(-1)

            if sw.shape[0] != X.shape[0]:
                raise ValueError(
                    f'sample_weight must have shape ({X.shape[0]},), got {sw.shape} instead'
                )
            if np.any(sw < 0):
                raise ValueError('sample_weight cannot contain negative values.')
            if not np.all(np.isfinite(sw)):
                raise ValueError('sample_weight must be finite.')

            if not np.all(sw == np.floor(sw)):
                raise ValueError(
                    'For sklearn-compatible sample_weight equivalence, sample_weight must be '
                    'nonnegative integers (0, 1, 2, ...).'
                )

            sw_int = sw.astype(int)
            total = int(sw_int.sum())
            if total <= 0:
                raise ValueError('sample_weight must sum to a positive value.')

            max_expanded = 5_000_000
            if total > max_expanded:
                raise ValueError(
                    f'Expanded dataset would have {total} rows, which is too large for exact '
                    f'sample_weight equivalence in memory.'
                )

            X = np.repeat(X, sw_int, axis=0)
            y = np.repeat(y, sw_int, axis=0)

        # --- RNG ---
        rng = check_random_state(self.random_state)

        # --- optional internal train/val split for early stopping ---
        X_val = y_val = None
        if self.early_stopping:
            X, y, X_val, y_val = self._train_val_split(X, y, rng, self.validation_fraction)

        # --- Dataset dimensions and effective batch size ---
        n_samples, n_features = X.shape
        bs = min(self.batch_size, n_samples)
        if bs <= 0:
            raise ValueError('batch_size must be a positive integer.')

        # --- Model parameter initialization ---
        beta, intercept = self._init_model_params(n_features, y)

        # --- Averaged parameters ---
        beta_avg = beta.copy()
        intercept_avg = intercept

        # --- AdaGrad state initialization ---
        G_beta = np.zeros_like(beta, dtype=self.dtype)
        G_intercept = self.dtype(0.0)
        eps = self.dtype(self.adagrad_eps)

        # --- Training diagnostics ---
        self.history_ = {
            'iter': [],
            'train_pinball_loss': [],
            'val_pinball_loss': [],
            'monitor_pinball_loss': [],
            'train_kkt_grad_inf': [],
            'monitor_kkt_grad_inf': [],
            'coef_path': [],
            'intercept_path': [],
            'monitor_name': monitor_name,
        }

        # --- "best" attributes ---
        self.best_val_loss_ = None
        self.best_iter_ = None
        self.best_train_loss_ = None

        # --- Early stopping state ---
        best_loss = None
        best_beta = None
        best_intercept = None
        no_improve_count = 0

        for t in range(1, self.max_iter + 1):
            idx = rng.choice(n_samples, size=bs, replace=False)
            X_batch, y_batch = X[idx], y[idx]

            if not self.use_adagrad:
                # --- Prox-SGD with diminishing step size step_size_t = base_lr / sqrt(t) ---
                step_size = self._sqrt_decay_lr(self.base_lr, t)

                beta, intercept = self._sgd_prox_step(
                    X_batch,
                    y_batch,
                    beta,
                    intercept,
                    step_size,
                    l1=self.l1,
                    l2=self.l2,
                )

            else:
                # --- AdaGrad update ---
                grad_beta, grad_intercept = self._batch_gradient(X_batch, y_batch, beta, intercept)

                G_beta += grad_beta**2
                G_intercept += self.dtype(grad_intercept**2)

                step_beta = self.base_lr / (np.sqrt(G_beta) + eps)
                step_intercept = self.base_lr / (np.sqrt(G_intercept) + eps)

                beta = beta - step_beta * grad_beta
                intercept = intercept - step_intercept * grad_intercept

                if self.l1 > 0 or self.l2 > 0:
                    beta = self._prox_elastic_net(beta, self.l1, self.l2, step_beta)

            # --- Parameter averaging (optional) ---
            if self.use_averaging:
                beta_avg = self._update_running_average(beta_avg, beta, t)
                intercept_avg = self._update_running_average(intercept_avg, intercept, t)
            else:
                beta_avg = beta
                intercept_avg = intercept

            # --- Periodic evaluation ---
            if t % self.eval_every == 0 or t == self.max_iter:
                eff_beta, eff_intercept = self._get_effective_params(
                    beta, intercept, beta_avg, intercept_avg
                )

                # Compute diagnostics first
                train_loss = self._pinball_loss_from_params(X, y, eff_beta, eff_intercept)

                val_loss = None
                if self.early_stopping:
                    val_loss = self._pinball_loss_from_params(X_val, y_val, eff_beta, eff_intercept)

                monitor_loss = None
                if record_monitor_loss and X_monitor is not None:
                    monitor_loss = self._pinball_loss_from_params(
                        X_monitor, y_monitor, eff_beta, eff_intercept
                    )

                train_kkt = None
                if record_train_kkt:
                    train_kkt = self._kkt_grad_inf(X, y, eff_beta, eff_intercept)

                monitor_kkt = None
                if record_monitor_kkt and X_monitor is not None:
                    monitor_kkt = self._kkt_grad_inf(
                        X_monitor, y_monitor, eff_beta, eff_intercept
                    )

                # Best-iteration bookkeeping for early stopping
                if self.early_stopping and val_loss is not None:
                    if best_loss is None or self._is_improvement(val_loss, best_loss):
                        self.best_iter_ = t
                        self.best_val_loss_ = val_loss
                        self.best_train_loss_ = train_loss

                # Append all diagnostics
                self.history_['iter'].append(t)
                self.history_['train_pinball_loss'].append(train_loss)
                self.history_['val_pinball_loss'].append(val_loss)
                self.history_['monitor_pinball_loss'].append(monitor_loss)
                self.history_['train_kkt_grad_inf'].append(train_kkt)
                self.history_['monitor_kkt_grad_inf'].append(monitor_kkt)

                if store_param_path:
                    self.history_['coef_path'].append(np.asarray(eff_beta, dtype=self.dtype).copy())
                    self.history_['intercept_path'].append(self.dtype(eff_intercept))
                else:
                    self.history_['coef_path'].append(None)
                    self.history_['intercept_path'].append(None)

                # Verbose printing
                if self.verbose:
                    msg = f"Iter {t}, train pinball loss = {train_loss:.6f}"
                    if val_loss is not None:
                        msg += f", val pinball loss = {val_loss:.6f}"
                    if monitor_loss is not None:
                        msg += f", {monitor_name} pinball loss = {monitor_loss:.6f}"
                    if train_kkt is not None:
                        msg += f", train KKT_inf = {train_kkt:.6e}"
                    if monitor_kkt is not None:
                        msg += f", {monitor_name} KKT_inf = {monitor_kkt:.6e}"
                    print(msg)

                # --- Early stopping update (still based on internal validation loss) ---
                if self.early_stopping:
                    best_loss, no_improve_count, best_beta, best_intercept, should_stop = (
                        self._early_stopping_step(
                            loss_value=val_loss,
                            best_loss=best_loss,
                            no_improve_count=no_improve_count,
                            best_beta=best_beta,
                            best_intercept=best_intercept,
                            current_beta=eff_beta,
                            current_intercept=eff_intercept,
                        )
                    )

                    if should_stop:
                        if self.verbose:
                            print(
                                f"Early stopping at iter {t}: no improvement "
                                f"for {self.n_iter_no_change} eval checks "
                                f"(best val loss={best_loss:.6f})."
                            )
                        break

        # --- Final parameter selection ---
        if self.early_stopping and self.restore_best_weights and best_beta is not None:
            beta_final, intercept_final = best_beta, best_intercept
        else:
            if self.use_averaging:
                beta_final, intercept_final = beta_avg, intercept_avg
            else:
                beta_final, intercept_final = beta, intercept

        self.coef_ = np.asarray(beta_final, dtype=self.dtype).reshape(-1)
        self.intercept_ = self.dtype(intercept_final)
        self.n_iter_ = t

        return self

    def predict(self, X):
        '''
        Predict targets using the fitted linear quantile regression model.

        Given a fitted model with learned parameters (coef_, intercept_), this returns
            y_pred = X @ coef_ + intercept_.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Feature matrix for which to generate predictions.

        Returns
        -------
        y_pred : ndarray of shape (n_samples,)
            Predicted values (estimated τ-quantile under the linear model).
        '''
        check_is_fitted(self, ['coef_', 'intercept_'])
        X = validate_data(self, X, accept_sparse=False, reset=False)
        X = np.asarray(X, dtype=self.dtype)
        y_pred = X @ self.coef_ + self.intercept_

        return np.asarray(y_pred).reshape(-1)

    def pinball_loss(self, X, y_true):
        '''
        Compute the mean pinball (check) loss of the fitted model on a dataset.

        For quantile level tau and residuals r_i = y_i - (x_i^T coef_ + intercept_),
        the pinball loss is
            tau * max(r_i, 0) + (1 - tau) * max(-r_i, 0).
        This method returns the empirical mean (1/n) * sum_i pinball_loss(r_i).

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Feature matrix.
        y_true : ndarray of shape (n_samples,)
            True target values.

        Returns
        -------
        loss : float
            Mean pinball loss of the fitted model on (X, y_true).
        '''
        check_is_fitted(self, ['coef_', 'intercept_'])
        X = validate_data(self, X, accept_sparse=False, reset=False)
        X = np.asarray(X, dtype=self.dtype)
        y_true = np.asarray(y_true, dtype=self.dtype).reshape(-1)

        return self._pinball_loss_from_params(X, y_true, self.coef_, self.intercept_)

# Metrics

In [ ]:
# Core Metrics
def pinball_loss(y_true, y_pred, tau):
    """
    Mean pinball loss for quantile regression.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    r = y_true - y_pred
    return np.mean(np.maximum(tau * r, (tau - 1) * r))


def rmse(y_true, y_pred):
    """
    Root mean squared error.
    """
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def coverage_rate(y_true, y_lower, y_upper):
    """
    Empirical coverage of prediction intervals.
    """
    y_true = np.asarray(y_true)
    return np.mean((y_true >= y_lower) & (y_true <= y_upper))


def mean_interval_width(y_lower, y_upper):
    """
    Average width of prediction intervals.
    """
    return np.mean(y_upper - y_lower)


def crossing_rate(y_lower, y_upper):
    """
    Fraction of samples where lower quantile exceeds upper quantile.
    """
    return np.mean(y_lower > y_upper)


def winkler_score(y_true, y_lower, y_upper, alpha):
    """
    Winkler score for prediction intervals.
    Lower is better.
    """
    y_true = np.asarray(y_true)
    width = y_upper - y_lower

    below = y_true < y_lower
    above = y_true > y_upper

    score = width.copy()
    score[below] += (2 / alpha) * (y_lower[below] - y_true[below])
    score[above] += (2 / alpha) * (y_true[above] - y_upper[above])

    return np.mean(score)



# Interval Metric Wrappers
def interval_metrics_basic(y_true, y_lower, y_upper):
    """
    Basic interval metrics: coverage, width, crossing.
    """
    return {
        "coverage": coverage_rate(y_true, y_lower, y_upper),
        "mean_width": mean_interval_width(y_lower, y_upper),
        "crossing_rate": crossing_rate(y_lower, y_upper),
    }


def interval_metrics_with_winkler(y_true, y_lower, y_upper, alpha):
    """
    Extended interval metrics including Winkler score.
    """
    metrics = interval_metrics_basic(y_true, y_lower, y_upper)
    metrics["winkler"] = winkler_score(y_true, y_lower, y_upper, alpha)
    return metrics



# KKT Stationarity Diagnostic
def kkt_grad_inf(X, y, coef, intercept, tau):
    """
    Infinity norm of stationarity residual:
        g = (1/n) X^T s,  where s_i = tau - 1{r_i < 0}

    This is a proxy for KKT optimality (exact when l1=0).
    """
    X = np.asarray(X)
    y = np.asarray(y)

    r = y - (X @ coef + intercept)
    s = tau - (r < 0).astype(float)

    g = (X.T @ s) / X.shape[0]
    return np.linalg.norm(g, ord=np.inf)


def kkt_stationarity_qr(X, y, coef, intercept, tau):
    """
    Returns both coefficient and intercept stationarity diagnostics.
    """
    X = np.asarray(X)
    y = np.asarray(y)

    r = y - (X @ coef + intercept)
    s = tau - (r < 0).astype(float)

    g = (X.T @ s) / X.shape[0]
    g0 = np.mean(s)  # intercept condition

    return {
        "kkt_grad_inf": np.linalg.norm(g, ord=np.inf),
        "kkt_intercept_abs": np.abs(g0),
    }

# Data Loaders

## Synthetic data

In [ ]:
# ================================
# Synthetic Data Generators
# ================================

from scipy.stats import norm, t, logistic, cauchy, gumbel_r


def sample_design_X(n, d, rng, covariance="toeplitz", rho=0.0):
    """
    Sample Gaussian design matrix with either identity or Toeplitz covariance.
    """
    if not (0 <= rho < 1):
        raise ValueError("rho must be in [0, 1).")

    if covariance == "identity" or rho == 0.0:
        Sigma = np.eye(d)
    elif covariance == "toeplitz":
        idx = np.arange(d)
        Sigma = rho ** np.abs(idx[:, None] - idx[None, :])
    else:
        raise ValueError(f"Unknown covariance type: {covariance}")

    L = np.linalg.cholesky(Sigma)
    Z = rng.standard_normal(size=(n, d))
    return Z @ L.T


def gen_synthetic_qr_data(
    n,
    d,
    rng,
    tau,
    rho=0.0,
    covariance="toeplitz",
    sparsity=0.0,
    noise_dist="gaussian",
    sigma=1.0,
    heteroskedastic=False,
    hetero_strength=0.5,
    t_df=3,
):
    """
    Generate synthetic quantile regression data.

    Model:
        y = X beta_star + eps

    The synthetic experiments in the results notebook do not vary sparsity,
    but use the default sparsity=0.0.
    """
    X = sample_design_X(n, d, rng, covariance=covariance, rho=rho)

    beta_star = np.zeros(d)
    k = max(1, int(sparsity * d))
    support = rng.choice(d, size=k, replace=False)
    beta_star[support] = rng.normal(0.0, 1.0, size=k)

    mu = X @ beta_star
    base_sigma = float(sigma)

    if heteroskedastic:
        scale = base_sigma * (1.0 + float(hetero_strength) * np.abs(mu))
    else:
        scale = base_sigma

    if noise_dist == "gaussian":
        eps = rng.normal(0.0, scale, size=n)
        q_shift = scale * norm.ppf(tau)

    elif noise_dist == "laplace":
        b = scale / np.sqrt(2)
        eps = rng.laplace(0.0, b, size=n)
        q_shift = b * np.log(tau / (1.0 - tau))

    elif noise_dist == "student_t":
        df = float(t_df)
        eps = rng.standard_t(df, size=n) * scale
        q_shift = scale * t.ppf(tau, df)

    elif noise_dist == "logistic":
        eps = rng.logistic(0.0, scale, size=n)
        q_shift = logistic.ppf(tau, loc=0.0, scale=scale)

    elif noise_dist == "cauchy":
        eps = rng.standard_cauchy(size=n) * scale
        q_shift = cauchy.ppf(tau, loc=0.0, scale=scale)

    elif noise_dist == "gumbel":
        eps = rng.gumbel(0.0, scale, size=n)
        q_shift = gumbel_r.ppf(tau, loc=0.0, scale=scale)

    else:
        raise ValueError(f"Unknown noise_dist: {noise_dist}")

    y = mu + eps
    y_true_tau = mu + q_shift

    if heteroskedastic:
        intercept_true = None
    else:
        intercept_true = float(np.asarray(q_shift).reshape(-1)[0])

    return X, y, beta_star, intercept_true, y_true_tau, mu


def true_quantile_from_components(
    X,
    beta_star,
    tau,
    *,
    noise_dist="gaussian",
    sigma=1.0,
    heteroskedastic=False,
    hetero_strength=0.5,
    t_df=3,
):
    """
    Compute the pointwise true conditional tau-quantile from X and beta_star.
    """
    X = np.asarray(X)
    beta_star = np.asarray(beta_star).reshape(-1)

    mu = X @ beta_star
    base_sigma = float(sigma)

    if heteroskedastic:
        scale = base_sigma * (1.0 + float(hetero_strength) * np.abs(mu))
    else:
        scale = base_sigma

    if noise_dist == "gaussian":
        q_shift = scale * norm.ppf(tau)

    elif noise_dist == "laplace":
        b = scale / np.sqrt(2)
        q_shift = b * np.log(tau / (1.0 - tau))

    elif noise_dist == "student_t":
        q_shift = scale * t.ppf(tau, float(t_df))

    elif noise_dist == "logistic":
        q_shift = logistic.ppf(tau, loc=0.0, scale=scale)

    elif noise_dist == "cauchy":
        q_shift = cauchy.ppf(tau, loc=0.0, scale=scale)

    elif noise_dist == "gumbel":
        q_shift = gumbel_r.ppf(tau, loc=0.0, scale=scale)

    else:
        raise ValueError(f"Unknown noise_dist: {noise_dist}")

    return mu + q_shift

## Benchmark data

In [ ]:
# Benchmark loaders
from sklearn.datasets import fetch_california_housing

def load_engel():
    """
    Load Engel food expenditure dataset from statsmodels.

    Returns
    -------
    X : ndarray, shape (n, 1)
    y : ndarray, shape (n,)
    feature_names : list[str]
    """
    data = sm.datasets.engel.load_pandas().data
    X = data[["income"]].to_numpy()
    y = data["foodexp"].to_numpy()
    return X, y, ["income"]


def load_abalone():
    """
    Load Abalone dataset from UCI-style URL or local fallback if already downloaded.

    Returns
    -------
    X : ndarray
    y : ndarray
    feature_names : list[str]
    """
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"

    cols = [
        "Sex", "Length", "Diameter", "Height", "WholeWeight",
        "ShuckedWeight", "VisceraWeight", "ShellWeight", "Rings"
    ]

    df = pd.read_csv(url, names=cols)
    df = pd.get_dummies(df, columns=["Sex"], drop_first=True)

    y = df["Rings"].to_numpy()
    X = df.drop(columns=["Rings"]).to_numpy()
    feature_names = df.drop(columns=["Rings"]).columns.tolist()

    return X, y, feature_names


def load_concrete():
    """
    Load Concrete Compressive Strength dataset.

    Returns
    -------
    X : ndarray
    y : ndarray
    feature_names : list[str]
    """
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

    df = pd.read_excel(url)

    y_col = df.columns[-1]
    y = df[y_col].to_numpy()
    X = df.drop(columns=[y_col]).to_numpy()
    feature_names = df.drop(columns=[y_col]).columns.tolist()

    return X, y, feature_names


def load_california_housing(remove_censored=True):
    """
    Load California Housing dataset.

    Parameters
    ----------
    remove_censored : bool
        If True, remove observations with target value equal to the censoring cap.

    Returns
    -------
    X : ndarray
    y : ndarray
    feature_names : list[str]
    """

    data = fetch_california_housing(as_frame=True)
    df = data.frame.copy()

    if remove_censored:
        df = df[df["MedHouseVal"] < 5.0].copy()

    y = df["MedHouseVal"].to_numpy()
    X = df.drop(columns=["MedHouseVal"]).to_numpy()
    feature_names = df.drop(columns=["MedHouseVal"]).columns.tolist()

    return X, y, feature_names


def load_dataset(name, **kwargs):
    """
    Unified dataset loader.

    Parameters
    ----------
    name : str
        One of {'engel', 'abalone', 'concrete', 'california'}.

    Returns
    -------
    X : ndarray
    y : ndarray
    feature_names : list[str]
    """
    name = name.lower()

    if name == "engel":
        return load_engel()

    if name == "abalone":
        return load_abalone()

    if name == "concrete":
        return load_concrete()

    if name in {"california", "california_housing", "cal_housing"}:
        return load_california_housing(**kwargs)

    raise ValueError(f"Unknown dataset name: {name}")

# Fit by method functions

## Standard results dictionary

In [ ]:
def _fit_result_dict(
    *,
    method,
    tau,
    model,
    coef,
    intercept,
    train_pred,
    test_pred,
    fit_seconds,
    n_iter=None,
    X_train=None,
    y_train=None,
    X_test=None,
    y_test=None,
):
    """
    Standardized result dictionary returned by all quantile-regression fitters.
    """
    result = {
        "method": method,
        "tau": tau,
        "model": model,
        "coef": coef,
        "beta": coef,  # alias for older CPS code
        "intercept": intercept,
        "train_pred": train_pred,
        "test_pred": test_pred,
        "yhat_train": train_pred,  # alias for older CPS code
        "yhat_test": test_pred,
        "fit_seconds": fit_seconds,
        "fit_time_sec": fit_seconds,
        "n_iter": n_iter,
    }

    if y_train is not None and train_pred is not None:
        result["train_pinball"] = pinball_loss(y_train, train_pred, tau)

    if y_test is not None and test_pred is not None:
        result["test_pinball"] = pinball_loss(y_test, test_pred, tau)

    if X_train is not None and y_train is not None:
        result["train_kkt_grad_inf"] = kkt_grad_inf(
            X_train, y_train, coef, intercept, tau
        )

    if X_test is not None and y_test is not None:
        result["test_kkt_grad_inf"] = kkt_grad_inf(
            X_test, y_test, coef, intercept, tau
        )

    return result

## Fit LP

In [ ]:
def fit_lp_quantile(
    X_train,
    y_train,
    X_test=None,
    y_test=None,
    tau=0.5,
    alpha=0.0,
    solver="highs",
    fit_intercept=True,
    solver_options=None,
):
    """
    Fit LP-based quantile regression using sklearn QuantileRegressor.
    """
    solver_options = {} if solver_options is None else solver_options

    model = QuantileRegressor(
        quantile=tau,
        alpha=alpha,
        solver=solver,
        fit_intercept=fit_intercept,
        solver_options=solver_options,
    )

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - t0

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test) if X_test is not None else None

    return _fit_result_dict(
        method="lp",
        tau=tau,
        model=model,
        coef=np.asarray(model.coef_),
        intercept=float(model.intercept_),
        train_pred=train_pred,
        test_pred=test_pred,
        fit_seconds=fit_seconds,
        n_iter=getattr(model, "n_iter_", None),
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
    )

## Fit IRLS

In [ ]:
def fit_irls_quantile(
    X_train,
    y_train,
    X_test=None,
    y_test=None,
    tau=0.5,
    max_iter=3_000,
    p_tol=1e-6,
):
    """
    Fit IRLS quantile regression using statsmodels QuantReg.
    """
    X_train_sm = sm.add_constant(X_train, has_constant="add")
    X_test_sm = sm.add_constant(X_test, has_constant="add") if X_test is not None else None

    model = sm.QuantReg(y_train, X_train_sm)

    t0 = time.perf_counter()
    res = model.fit(q=tau, max_iter=max_iter, p_tol=p_tol)
    fit_seconds = time.perf_counter() - t0

    params = np.asarray(res.params)
    intercept = float(params[0])
    coef = params[1:]

    train_pred = X_train_sm @ params
    test_pred = X_test_sm @ params if X_test_sm is not None else None

    n_iter = getattr(res, "iterations", None)

    return _fit_result_dict(
        method="irls",
        tau=tau,
        model=res,
        coef=coef,
        intercept=intercept,
        train_pred=train_pred,
        test_pred=test_pred,
        fit_seconds=fit_seconds,
        n_iter=n_iter,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
    )

## Fit proxSGD

In [ ]:
def fit_proxsgd_quantile(
    X_train,
    y_train,
    X_test=None,
    y_test=None,
    tau=0.5,
    sgd_kwargs=None,
):
    """
    Fit proxSGD quantile regression using SGDQuantileRegressor.
    """
    sgd_kwargs = {} if sgd_kwargs is None else dict(sgd_kwargs)
    sgd_kwargs["quantile"] = tau

    model = SGDQuantileRegressor(**sgd_kwargs)

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - t0

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test) if X_test is not None else None

    return _fit_result_dict(
        method="proxsgd",
        tau=tau,
        model=model,
        coef=np.asarray(model.coef_),
        intercept=float(model.intercept_),
        train_pred=train_pred,
        test_pred=test_pred,
        fit_seconds=fit_seconds,
        n_iter=getattr(model, "n_iter_", None),
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
    )

## Fit by method dispatcher

In [ ]:
def fit_quantile_method(
    method,
    X_train,
    y_train,
    X_test=None,
    y_test=None,
    tau=0.5,
    *,
    lp_alpha=0.0,
    lp_solver="highs",
    lp_solver_options=None,
    irls_max_iter=10_000,
    irls_p_tol=1e-6,
    sgd_kwargs=None,
):
    """
    Unified dispatcher for LP, IRLS, and proxSGD quantile-regression fits.
    """
    method_key = method.lower()

    if method_key in {"lp", "sklearn", "sklearn_lp"}:
        return fit_lp_quantile(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            tau=tau,
            alpha=lp_alpha,
            solver=lp_solver,
            solver_options=lp_solver_options,
        )

    if method_key in {"irls", "statsmodels", "quantreg"}:
        return fit_irls_quantile(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            tau=tau,
            max_iter=irls_max_iter,
            p_tol=irls_p_tol,
        )

    if method_key in {"proxsgd", "sgd", "prox_sgd"}:
        return fit_proxsgd_quantile(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            tau=tau,
            sgd_kwargs=sgd_kwargs,
        )

    raise ValueError(
        f"Unknown method '{method}'. Expected one of: 'lp', 'irls', 'proxsgd'."
    )

# Synthetic experiment runners

## Helpers

In [ ]:
def make_sgd_kwargs(
    *,
    tau,
    base_lr=0.5,
    batch_size=256,
    max_iter=3000,
    l1=0.0,
    l2=0.0,
    eval_every=500,
    use_adagrad=True,
    adagrad_eps=1e-8,
    use_averaging=True,
    dtype=np.float64,
    random_state=None,
    verbose=False,
    **extra_kwargs,
):
    """
    Build kwargs for SGDQuantileRegressor in one consistent place.
    """
    kwargs = dict(
        quantile=tau,
        max_iter=max_iter,
        base_lr=base_lr,
        batch_size=batch_size,
        l1=l1,
        l2=l2,
        eval_every=eval_every,
        use_adagrad=use_adagrad,
        adagrad_eps=adagrad_eps,
        use_averaging=use_averaging,
        dtype=dtype,
        random_state=random_state,
        verbose=verbose,
    )

    kwargs.update(extra_kwargs)
    return kwargs


def _get_row_value(row, name, default=None):
    """
    Helper for reading experiment-table rows with defaults.
    """
    return row[name] if name in row and pd.notna(row[name]) else default


def _method_list_from_row(row, default_methods=("lp", "irls", "proxsgd")):
    """
    Read methods from an experiment-table row, if provided.
    """
    methods = _get_row_value(row, "methods", default_methods)

    if isinstance(methods, str):
        methods = [m.strip() for m in methods.split(",")]

    return tuple(methods)

## Preprocess

In [ ]:
def split_and_standardize(X, y, test_size=1000, random_state=None):
    """
    Train/test split followed by standardization fit on the training data.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, scaler

## Fit by quantile

In [ ]:
def run_synthetic_point_experiment(
    row,
    seed,
    methods=("lp", "irls", "proxsgd"),
    test_size=1000,
    default_sgd_kwargs=None,
):
    """
    Run one synthetic point-quantile experiment for one experiment-table row and seed.

    Returns one row per method.
    """
    default_sgd_kwargs = {} if default_sgd_kwargs is None else dict(default_sgd_kwargs)

    experiment = _get_row_value(row, "experiment", "synthetic")
    experiment_group = _get_row_value(row, "experiment_group", None)

    n = int(_get_row_value(row, "n", 10_000))
    d = int(_get_row_value(row, "d", 100))
    tau = float(_get_row_value(row, "tau", 0.5))

    rho = float(_get_row_value(row, "rho", 0.0))
    covariance = _get_row_value(row, "covariance", "toeplitz")
    noise_dist = _get_row_value(row, "noise_dist", "gaussian")
    sigma = float(_get_row_value(row, "sigma", 1.0))
    heteroskedastic = bool(_get_row_value(row, "heteroskedastic", False))
    hetero_strength = float(_get_row_value(row, "hetero_strength", 0.5))
    t_df = float(_get_row_value(row, "t_df", 3))

    sparsity = float(_get_row_value(row, "sparsity", 0.2))

    lp_alpha = float(_get_row_value(row, "lp_alpha", 0.0))
    l1 = float(_get_row_value(row, "l1", lp_alpha))
    l2 = float(_get_row_value(row, "l2", 0.0))

    base_lr = float(_get_row_value(row, "base_lr", default_sgd_kwargs.get("base_lr", 0.5)))
    batch_size = int(_get_row_value(row, "batch_size", default_sgd_kwargs.get("batch_size", 256)))
    max_iter = int(_get_row_value(row, "max_iter", default_sgd_kwargs.get("max_iter", 3000)))
    eval_every = int(_get_row_value(row, "eval_every", default_sgd_kwargs.get("eval_every", 500)))

    rng = np.random.default_rng(seed)

    X, y, beta_star, intercept_true, y_true_tau, mu = gen_synthetic_qr_data(
        n=n,
        d=d,
        rng=rng,
        tau=tau,
        rho=rho,
        covariance=covariance,
        sparsity=sparsity,
        noise_dist=noise_dist,
        sigma=sigma,
        heteroskedastic=heteroskedastic,
        hetero_strength=hetero_strength,
        t_df=t_df,
    )

    X_train, X_test, y_train, y_test, scaler = split_and_standardize(
        X,
        y,
        test_size=test_size,
        random_state=seed,
    )

    results = []

    for method in methods:
        method_key = method.lower()

        if method_key == "irls" and lp_alpha > 0:
            continue

        sgd_kwargs = make_sgd_kwargs(
            tau=tau,
            base_lr=base_lr,
            batch_size=batch_size,
            max_iter=max_iter,
            l1=l1,
            l2=l2,
            eval_every=eval_every,
            random_state=seed,
            **{
                k: v
                for k, v in default_sgd_kwargs.items()
                if k not in {
                    "quantile", "base_lr", "batch_size", "max_iter",
                    "l1", "l2", "eval_every", "random_state"
                }
            },
        )

        fit = fit_quantile_method(
            method=method_key,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            tau=tau,
            lp_alpha=lp_alpha,
            sgd_kwargs=sgd_kwargs,
        )

        results.append({
            "experiment": experiment,
            "experiment_group": experiment_group,
            "seed": seed,
            "method": fit["method"],
            "tau": tau,
            "n": n,
            "n_train": X_train.shape[0],
            "n_test": X_test.shape[0],
            "d": d,
            "rho": rho,
            "covariance": covariance,
            "noise_dist": noise_dist,
            "sigma": sigma,
            "heteroskedastic": heteroskedastic,
            "hetero_strength": hetero_strength,
            "t_df": t_df,
            "sparsity": sparsity,
            "lp_alpha": lp_alpha,
            "l1": l1,
            "l2": l2,
            "base_lr": base_lr,
            "batch_size": batch_size,
            "max_iter": max_iter,
            "fit_seconds": fit["fit_seconds"],
            "n_iter": fit["n_iter"],
            "train_pinball": fit.get("train_pinball", np.nan),
            "test_pinball": fit.get("test_pinball", np.nan),
            "train_kkt_grad_inf": fit.get("train_kkt_grad_inf", np.nan),
            "test_kkt_grad_inf": fit.get("test_kkt_grad_inf", np.nan),
        })

    return pd.DataFrame(results)

## Fit interval

In [ ]:
def run_synthetic_interval_experiment(
    row,
    seed,
    methods=("lp", "irls", "proxsgd"),
    test_size=1000,
    default_sgd_kwargs=None,
):
    """
    Run one synthetic interval experiment for one experiment-table row and seed.

    Fits lower and upper quantile models for each method, then computes
    interval metrics.
    """
    default_sgd_kwargs = {} if default_sgd_kwargs is None else dict(default_sgd_kwargs)

    experiment = _get_row_value(row, "experiment", "synthetic")
    experiment_group = _get_row_value(row, "experiment_group", None)

    n = int(_get_row_value(row, "n", 10_000))
    d = int(_get_row_value(row, "d", 100))

    lower_tau = float(_get_row_value(row, "lower_tau", 0.025))
    upper_tau = float(_get_row_value(row, "upper_tau", 0.975))
    alpha_interval = upper_tau - lower_tau
    miscoverage = 1.0 - alpha_interval

    rho = float(_get_row_value(row, "rho", 0.0))
    covariance = _get_row_value(row, "covariance", "toeplitz")
    noise_dist = _get_row_value(row, "noise_dist", "gaussian")
    sigma = float(_get_row_value(row, "sigma", 1.0))
    heteroskedastic = bool(_get_row_value(row, "heteroskedastic", False))
    hetero_strength = float(_get_row_value(row, "hetero_strength", 0.5))
    t_df = float(_get_row_value(row, "t_df", 3))

    sparsity = float(_get_row_value(row, "sparsity", 0.2))

    lp_alpha = float(_get_row_value(row, "lp_alpha", 0.0))
    l1 = float(_get_row_value(row, "l1", lp_alpha))
    l2 = float(_get_row_value(row, "l2", 0.0))

    base_lr = float(_get_row_value(row, "base_lr", default_sgd_kwargs.get("base_lr", 0.5)))
    batch_size = int(_get_row_value(row, "batch_size", default_sgd_kwargs.get("batch_size", 256)))
    max_iter = int(_get_row_value(row, "max_iter", default_sgd_kwargs.get("max_iter", 3000)))
    eval_every = int(_get_row_value(row, "eval_every", default_sgd_kwargs.get("eval_every", 500)))

    rng = np.random.default_rng(seed)

    X, y, beta_star, _, _, mu = gen_synthetic_qr_data(
        n=n,
        d=d,
        rng=rng,
        tau=0.5,
        rho=rho,
        covariance=covariance,
        sparsity=sparsity,
        noise_dist=noise_dist,
        sigma=sigma,
        heteroskedastic=heteroskedastic,
        hetero_strength=hetero_strength,
        t_df=t_df,
    )

    X_train, X_test, y_train, y_test, scaler = split_and_standardize(
        X,
        y,
        test_size=test_size,
        random_state=seed,
    )

    results = []

    for method in methods:
        method_key = method.lower()

        if method_key == "irls" and lp_alpha > 0:
            continue

        fits = {}

        for tau in (lower_tau, upper_tau):
            sgd_kwargs = make_sgd_kwargs(
                tau=tau,
                base_lr=base_lr,
                batch_size=batch_size,
                max_iter=max_iter,
                l1=l1,
                l2=l2,
                eval_every=eval_every,
                random_state=seed,
                **{
                    k: v
                    for k, v in default_sgd_kwargs.items()
                    if k not in {
                        "quantile", "base_lr", "batch_size", "max_iter",
                        "l1", "l2", "eval_every", "random_state"
                    }
                },
            )

            fits[tau] = fit_quantile_method(
                method=method_key,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                tau=tau,
                lp_alpha=lp_alpha,
                sgd_kwargs=sgd_kwargs,
            )

        lower_fit = fits[lower_tau]
        upper_fit = fits[upper_tau]

        y_lower = lower_fit["test_pred"]
        y_upper = upper_fit["test_pred"]

        interval = interval_metrics_with_winkler(
            y_test,
            y_lower,
            y_upper,
            alpha=miscoverage,
        )

        results.append({
            "experiment": experiment,
            "experiment_group": experiment_group,
            "seed": seed,
            "method": method_key,
            "lower_tau": lower_tau,
            "upper_tau": upper_tau,
            "nominal_coverage": alpha_interval,
            "n": n,
            "n_train": X_train.shape[0],
            "n_test": X_test.shape[0],
            "d": d,
            "rho": rho,
            "covariance": covariance,
            "noise_dist": noise_dist,
            "sigma": sigma,
            "heteroskedastic": heteroskedastic,
            "hetero_strength": hetero_strength,
            "t_df": t_df,
            "sparsity": sparsity,
            "lp_alpha": lp_alpha,
            "l1": l1,
            "l2": l2,
            "base_lr": base_lr,
            "batch_size": batch_size,
            "max_iter": max_iter,
            "fit_seconds_lower": lower_fit["fit_seconds"],
            "fit_seconds_upper": upper_fit["fit_seconds"],
            "fit_seconds": lower_fit["fit_seconds"] + upper_fit["fit_seconds"],
            "n_iter_lower": lower_fit["n_iter"],
            "n_iter_upper": upper_fit["n_iter"],
            "lower_test_pinball": lower_fit.get("test_pinball", np.nan),
            "upper_test_pinball": upper_fit.get("test_pinball", np.nan),
            "mean_test_pinball": np.nanmean([
                lower_fit.get("test_pinball", np.nan),
                upper_fit.get("test_pinball", np.nan),
            ]),
            "lower_train_kkt_grad_inf": lower_fit.get("train_kkt_grad_inf", np.nan),
            "upper_train_kkt_grad_inf": upper_fit.get("train_kkt_grad_inf", np.nan),
            "coverage": interval["coverage"],
            "mean_width": interval["mean_width"],
            "crossing_rate": interval["crossing_rate"],
            "winkler": interval["winkler"],
        })

    return pd.DataFrame(results)

## Experiment by table

In [ ]:
def run_synthetic_experiment_table(
    experiment_table,
    *,
    mode="interval",
    methods=("lp", "irls", "proxsgd"),
    seeds=range(5),
    test_size=1000,
    default_sgd_kwargs=None,
):
    """
    Run a full synthetic experiment table.

    Parameters
    ----------
    experiment_table : pandas.DataFrame
        One row per synthetic regime/configuration.
    mode : {'point', 'interval'}
        Whether to fit one tau per row or lower/upper quantiles per row.
    methods : tuple[str]
        Methods to evaluate. Options: 'lp', 'irls', 'proxsgd'.
    seeds : iterable
        Random seeds.
    test_size : int or float
        Test-set size passed to train_test_split.
    default_sgd_kwargs : dict or None
        Default proxSGD keyword arguments.

    Returns
    -------
    results : pandas.DataFrame
        Tidy synthetic experiment results.
    """
    all_results = []

    for _, row in experiment_table.iterrows():
        row_methods = _method_list_from_row(row, default_methods=methods)

        for seed in seeds:
            if mode == "point":
                out = run_synthetic_point_experiment(
                    row=row,
                    seed=seed,
                    methods=row_methods,
                    test_size=test_size,
                    default_sgd_kwargs=default_sgd_kwargs,
                )

            elif mode == "interval":
                out = run_synthetic_interval_experiment(
                    row=row,
                    seed=seed,
                    methods=row_methods,
                    test_size=test_size,
                    default_sgd_kwargs=default_sgd_kwargs,
                )

            else:
                raise ValueError("mode must be either 'point' or 'interval'.")

            all_results.append(out)

    return pd.concat(all_results, ignore_index=True)

# Benchmark experiment runners

## Preprocess

In [ ]:
def prepare_benchmark_split(
    dataset,
    *,
    test_size=0.2,
    random_state=33,
    scale=True,
    loader_kwargs=None,
):
    """
    Load a benchmark dataset, create a train/test split, and optionally standardize X.

    Returns
    -------
    dict
        Contains X_train, X_test, y_train, y_test, scaler, feature_names.
    """
    loader_kwargs = {} if loader_kwargs is None else dict(loader_kwargs)

    X, y, feature_names = load_dataset(dataset, **loader_kwargs)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
    )

    scaler = None
    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    return {
        "dataset": dataset,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": scaler,
        "feature_names": feature_names,
    }

## Fit by quantile

In [ ]:
def run_benchmark_quantile_fits(
    dataset,
    *,
    taus=(0.1, 0.5, 0.9),
    methods=("lp", "irls", "proxsgd"),
    test_size=0.2,
    random_state=33,
    scale=True,
    loader_kwargs=None,
    lp_alpha=0.0,
    default_sgd_kwargs=None,
    irls_max_iter=10_000,
    irls_p_tol=1e-6,
):
    """
    Fit each method at each tau for one benchmark dataset.

    Returns
    -------
    fit_results : dict
        Nested dictionary fit_results[method][tau] = fit result dict.
    point_rows : pandas.DataFrame
        One row per dataset/method/tau.
    split : dict
        Train/test split and metadata.
    """
    default_sgd_kwargs = {} if default_sgd_kwargs is None else dict(default_sgd_kwargs)

    split = prepare_benchmark_split(
        dataset,
        test_size=test_size,
        random_state=random_state,
        scale=scale,
        loader_kwargs=loader_kwargs,
    )

    X_train = split["X_train"]
    X_test = split["X_test"]
    y_train = split["y_train"]
    y_test = split["y_test"]

    fit_results = {}
    rows = []

    for method in methods:
        method_key = method.lower()

        if method_key == "irls" and lp_alpha > 0:
            continue

        fit_results[method_key] = {}

        for tau in taus:
            tau = float(tau)

            sgd_kwargs = dict(default_sgd_kwargs)
            sgd_kwargs["quantile"] = tau
            sgd_kwargs.setdefault("random_state", random_state)

            fit = fit_quantile_method(
                method=method_key,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                tau=tau,
                lp_alpha=lp_alpha,
                irls_max_iter=irls_max_iter,
                irls_p_tol=irls_p_tol,
                sgd_kwargs=sgd_kwargs,
            )

            fit_results[method_key][tau] = fit

            rows.append({
                "dataset": dataset,
                "random_state": random_state,
                "method": fit["method"],
                "tau": tau,
                "n_train": X_train.shape[0],
                "n_test": X_test.shape[0],
                "d": X_train.shape[1],
                "lp_alpha": lp_alpha,
                "l1": sgd_kwargs.get("l1", lp_alpha),
                "base_lr": sgd_kwargs.get("base_lr", np.nan),
                "batch_size": sgd_kwargs.get("batch_size", np.nan),
                "max_iter": sgd_kwargs.get("max_iter", np.nan),
                "fit_seconds": fit["fit_seconds"],
                "n_iter": fit["n_iter"],
                "train_pinball": fit.get("train_pinball", np.nan),
                "test_pinball": fit.get("test_pinball", np.nan),
                "train_kkt_grad_inf": fit.get("train_kkt_grad_inf", np.nan),
                "test_kkt_grad_inf": fit.get("test_kkt_grad_inf", np.nan),
                "rmse": rmse(y_test, fit["test_pred"]),
            })

    return fit_results, pd.DataFrame(rows), split

## Interval

In [ ]:
def run_benchmark_interval_experiment(
    dataset,
    *,
    lower_tau=0.1,
    upper_tau=0.9,
    methods=("lp", "irls", "proxsgd"),
    test_size=0.2,
    random_state=33,
    scale=True,
    loader_kwargs=None,
    lp_alpha=0.0,
    default_sgd_kwargs=None,
    irls_max_iter=10_000,
    irls_p_tol=1e-6,
):
    """
    Run benchmark interval experiment for one dataset.

    Fits lower, median, and upper quantiles when available, and reports:
    pinball loss, RMSE at tau=0.5, coverage, width, crossing, and Winkler.
    """
    taus = sorted(set([float(lower_tau), 0.5, float(upper_tau)]))

    fit_results, point_df, split = run_benchmark_quantile_fits(
        dataset,
        taus=taus,
        methods=methods,
        test_size=test_size,
        random_state=random_state,
        scale=scale,
        loader_kwargs=loader_kwargs,
        lp_alpha=lp_alpha,
        default_sgd_kwargs=default_sgd_kwargs,
        irls_max_iter=irls_max_iter,
        irls_p_tol=irls_p_tol,
    )

    y_test = split["y_test"]
    miscoverage = 1.0 - (upper_tau - lower_tau)

    rows = []

    for method_key, fits_by_tau in fit_results.items():
        if lower_tau not in fits_by_tau or upper_tau not in fits_by_tau:
            continue

        lower_fit = fits_by_tau[float(lower_tau)]
        upper_fit = fits_by_tau[float(upper_tau)]
        median_fit = fits_by_tau.get(0.5, None)

        y_lower = lower_fit["test_pred"]
        y_upper = upper_fit["test_pred"]

        interval = interval_metrics_with_winkler(
            y_test,
            y_lower,
            y_upper,
            alpha=miscoverage,
        )

        rows.append({
            "dataset": dataset,
            "random_state": random_state,
            "method": method_key,
            "lower_tau": float(lower_tau),
            "upper_tau": float(upper_tau),
            "nominal_coverage": float(upper_tau - lower_tau),
            "n_train": split["X_train"].shape[0],
            "n_test": split["X_test"].shape[0],
            "d": split["X_train"].shape[1],
            "lp_alpha": lp_alpha,
            "l1": lower_fit.get("model").l1 if hasattr(lower_fit.get("model"), "l1") else lp_alpha,
            "fit_seconds": lower_fit["fit_seconds"] + upper_fit["fit_seconds"],
            "fit_seconds_lower": lower_fit["fit_seconds"],
            "fit_seconds_upper": upper_fit["fit_seconds"],
            "lower_test_pinball": lower_fit.get("test_pinball", np.nan),
            "upper_test_pinball": upper_fit.get("test_pinball", np.nan),
            "mean_test_pinball": np.nanmean([
                lower_fit.get("test_pinball", np.nan),
                upper_fit.get("test_pinball", np.nan),
            ]),
            "rmse": rmse(y_test, median_fit["test_pred"]) if median_fit is not None else np.nan,
            "coverage": interval["coverage"],
            "mean_width": interval["mean_width"],
            "crossing_rate": interval["crossing_rate"],
            "winkler": interval["winkler"],
            "lower_train_kkt_grad_inf": lower_fit.get("train_kkt_grad_inf", np.nan),
            "upper_train_kkt_grad_inf": upper_fit.get("train_kkt_grad_inf", np.nan),
        })

    return pd.DataFrame(rows), point_df, fit_results, split

## Experiment by table

In [ ]:
def run_benchmark_experiment_table(
    experiment_table,
    *,
    methods=("lp", "irls", "proxsgd"),
    test_size=0.2,
    default_sgd_kwargs=None,
    scale=True,
):
    """
    Run benchmark interval experiments from an experiment table.

    Expected useful columns include:
        dataset, lower_tau, upper_tau, random_state, lp_alpha, l1,
        test_size, methods
    """
    all_interval = []
    all_point = []

    for _, row in experiment_table.iterrows():
        dataset = _get_row_value(row, "dataset")
        if dataset is None:
            raise ValueError("Each benchmark experiment row must include a 'dataset' column.")

        row_methods = _method_list_from_row(row, default_methods=methods)

        lower_tau = float(_get_row_value(row, "lower_tau", 0.1))
        upper_tau = float(_get_row_value(row, "upper_tau", 0.9))
        random_state = int(_get_row_value(row, "random_state", 123))
        row_test_size = _get_row_value(row, "test_size", test_size)
        lp_alpha = float(_get_row_value(row, "lp_alpha", 0.0))
        l1 = float(_get_row_value(row, "l1", lp_alpha))

        row_sgd_kwargs = {} if default_sgd_kwargs is None else dict(default_sgd_kwargs)
        row_sgd_kwargs["l1"] = l1

        for key in ["base_lr", "batch_size", "max_iter", "eval_every", "l2"]:
            if key in row and pd.notna(row[key]):
                row_sgd_kwargs[key] = row[key]

        interval_df, point_df, _, _ = run_benchmark_interval_experiment(
            dataset=dataset,
            lower_tau=lower_tau,
            upper_tau=upper_tau,
            methods=row_methods,
            test_size=row_test_size,
            random_state=random_state,
            scale=scale,
            lp_alpha=lp_alpha,
            default_sgd_kwargs=row_sgd_kwargs,
        )

        for col in ["experiment", "experiment_group"]:
            if col in row:
                interval_df[col] = row[col]
                point_df[col] = row[col]

        all_interval.append(interval_df)
        all_point.append(point_df)

    return {
        "interval": pd.concat(all_interval, ignore_index=True),
        "point": pd.concat(all_point, ignore_index=True),
    }

# CPS experiment runners

## Preprocess

In [ ]:
def prepare_cps_features(df):
    """
    Construct CPS feature matrix exactly as used in the paper.

    Model:
        log_wage ~ AGE + AGE^2 + EDUC
                   + SEX + RACE + HISPAN + REGION
                   + STATE + OCC + IND
    """
    df = df.copy()

    y = df["log_wage"].to_numpy()

    df["AGE2"] = df["AGE"] ** 2

    X_cont = df[["AGE", "AGE2", "EDUC"]]

    X_cat = pd.get_dummies(
        df[
            [
                "SEX",
                "RACE",
                "HISPAN",
                "REGION",
                "STATE",
                "OCC",
                "IND",
            ]
        ],
        drop_first=True,
    )

    X = pd.concat([X_cont, X_cat], axis=1)

    return X.to_numpy(), y, X.columns.tolist()

In [ ]:
def make_cps_train_test_split(
    df,
    *,
    test_size=10_000,
    random_state=123,
):
    """
    Create CPS train/test split with fixed test size.
    """
    X, y, feature_names = prepare_cps_features(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return {
        "X_train_full": X_train,
        "y_train_full": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "scaler": scaler,
        "feature_names": feature_names,
    }

## Fit by quantile

In [ ]:
def run_cps_quantile_fits(
    X_train,
    y_train,
    X_test,
    y_test,
    *,
    taus=(0.1, 0.5, 0.9),
    methods=("lp", "irls", "proxsgd"),
    lp_alpha=0.0,
    lp_solver="highs",
    lp_solver_options=None,
    irls_max_iter=3_000,
    irls_p_tol=1e-6,
    sgd_kwargs=None,
    seed=123,
):
    sgd_kwargs = {} if sgd_kwargs is None else dict(sgd_kwargs)

    fit_results = {}

    for method in methods:
        method_key = method.lower()

        if method_key == "irls" and lp_alpha > 0:
            continue

        fit_results[method_key] = {}

        for tau in taus:
            tau = float(tau)

            kwargs = dict(sgd_kwargs)
            kwargs["quantile"] = tau
            kwargs.setdefault("random_state", seed)

            fit = fit_quantile_method(
                method=method_key,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                tau=tau,
                lp_alpha=lp_alpha,
                lp_solver=lp_solver,
                lp_solver_options=lp_solver_options,
                irls_max_iter=irls_max_iter,
                irls_p_tol=irls_p_tol,
                sgd_kwargs=kwargs,
            )

            fit_results[method_key][tau] = fit

    return fit_results

## Scaling experiment

In [ ]:
def run_cps_scaling_experiment(
    df,
    *,
    N_SUB_list,
    taus=(0.1, 0.5, 0.9),
    methods=("lp", "irls", "proxsgd"),
    seed=123,
    test_size=10_000,
    lp_alpha=0.0,
    lp_solver="highs",
    lp_solver_options=None,
    irls_max_iter=3_000,
    irls_p_tol=1e-6,
    sgd_kwargs=None,
):
    base = make_cps_train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
    )

    X_full = base["X_train_full"]
    y_full = base["y_train_full"]
    X_test = base["X_test"]
    y_test = base["y_test"]

    results = []

    for N_SUB in N_SUB_list:
        X_train = X_full[:N_SUB]
        y_train = y_full[:N_SUB]

        fits = run_cps_quantile_fits(
            X_train,
            y_train,
            X_test,
            y_test,
            taus=taus,
            methods=methods,
            lp_alpha=lp_alpha,
            lp_solver=lp_solver,
            lp_solver_options=lp_solver_options,
            irls_max_iter=irls_max_iter,
            irls_p_tol=irls_p_tol,
            sgd_kwargs=sgd_kwargs,
            seed=seed,
        )

        for method, tau_dict in fits.items():
            lower = tau_dict.get(0.1)
            median = tau_dict.get(0.5)
            upper = tau_dict.get(0.9)

            if lower is None or upper is None:
                continue

            y_lower = lower["test_pred"]
            y_upper = upper["test_pred"]

            interval = interval_metrics_with_winkler(
                y_test,
                y_lower,
                y_upper,
                alpha=0.2,
            )

            results.append({
                "method": method,
                "N_SUB": N_SUB,
                "seed": seed,
                "tau": 0.5,
                "n_train": N_SUB,
                "n_test": len(y_test),
                "d": X_train.shape[1],
                "fit_seconds": sum(
                    tau_dict[t]["fit_seconds"] for t in tau_dict
                ),
                "coverage": interval["coverage"],
                "mean_width": interval["mean_width"],
                "winkler": interval["winkler"],
                "rmse": rmse(y_test, median["test_pred"]) if median else np.nan,
            })

    return pd.DataFrame(results)

# Summary functions

## Helpers

In [ ]:
def format_mean_sd_pm(mean, sd, digits=2):
    if pd.isna(mean):
        return ""
    if pd.isna(sd):
        return f"{mean:.{digits}f}"

    return rf"{mean:.{digits}f} $\pm$ {sd:.{digits}f}"


def summarize_by_mean_sd(
    df,
    group_cols,
    metric_cols,
    digits=4,
):
    """
    Summarize selected metrics by mean and standard deviation.

    Returns one row per group, with entries formatted as mean ± sd.
    """
    summary = (
        df.groupby(group_cols, dropna=False)[metric_cols]
        .agg(["mean", "std"])
        .reset_index()
    )

    # Flatten multi-index columns
    summary.columns = [
        "_".join(col).strip("_") if isinstance(col, tuple) else col
        for col in summary.columns
    ]

    out = summary[group_cols].copy()

    for metric in metric_cols:
        out[metric] = [
            format_mean_sd_pm(m, s, digits=digits)
            for m, s in zip(summary[f"{metric}_mean"], summary[f"{metric}_std"])
        ]

    return out

def dataframe_to_latex_table(
    df,
    *,
    caption=None,
    label=None,
    index=False,
    column_format=None,
    float_format=None,
    escape=False,
):
    """
    Convert a summary DataFrame to LaTeX.

    This is intentionally thin so that paper-specific formatting can remain
    in the results notebook.
    """
    return df.to_latex(
        index=index,
        caption=caption,
        label=label,
        column_format=column_format,
        float_format=float_format,
        escape=escape,
    )

## Synthetic data

In [ ]:
def summarize_synthetic_intervals(
    df,
    *,
    group_cols=("experiment", "method"),
    metric_cols=(
        "mean_test_pinball",
        "coverage",
        "mean_width",
        "crossing_rate",
        "fit_seconds",
    ),
    digits=4,
):
    """
    Summarize synthetic interval experiments using mean ± one standard deviation.

    Parameters
    ----------
    df : pandas.DataFrame
        Output from run_synthetic_experiment_table(..., mode="interval").
    group_cols : tuple[str]
        Columns to group by, typically ("experiment", "method").
    metric_cols : tuple[str]
        Numeric metric columns to summarize.
    digits : int
        Number of decimal places.

    Returns
    -------
    pandas.DataFrame
        Summary table with formatted mean ± sd entries.
    """
    return summarize_by_mean_sd(
        df,
        group_cols=list(group_cols),
        metric_cols=list(metric_cols),
        digits=digits,
    )

## Benchmark

In [ ]:
def summarize_benchmark_intervals(
    df,
    *,
    group_cols=("dataset", "method"),
    metric_cols=(
        "mean_test_pinball",
        "rmse",
        "coverage",
        "mean_width",
        "fit_seconds",
    ),
    digits=4,
):
    """
    Summarize benchmark interval experiments.
    """
    group_cols = list(group_cols)
    metric_cols = list(metric_cols)

    return summarize_by_mean_sd(
        df,
        group_cols=group_cols,
        metric_cols=metric_cols,
        digits=digits,
    )


def summarize_benchmark_point(
    df,
    *,
    group_cols=("dataset", "method", "tau"),
    metric_cols=(
        "test_pinball",
        "rmse",
        "train_kkt_grad_inf",
        "fit_seconds",
    ),
    digits=4,
):
    """
    Summarize benchmark point-quantile fits.
    """
    group_cols = list(group_cols)
    metric_cols = list(metric_cols)

    return summarize_by_mean_sd(
        df,
        group_cols=group_cols,
        metric_cols=metric_cols,
        digits=digits,
    )

In [ ]:
def summarize_convergence_results(curves, refs):
    """
    Summarize final proxSGD losses and LP/IRLS reference losses.
    """
    final_curves = (
        curves
        .sort_values(["dataset", "seed", "split", "iteration"])
        .groupby(["dataset", "seed", "split"], as_index=False)
        .tail(1)
    )

    prox_summary = (
        final_curves
        .groupby(["dataset", "split"])["pinball_loss"]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
        .rename(
            columns={
                "mean": "proxSGD_final_mean",
                "std": "proxSGD_final_std",
                "min": "proxSGD_final_min",
                "max": "proxSGD_final_max",
            }
        )
    )

    ref_summary = (
        refs
        .groupby(["dataset", "method", "split"])["pinball_loss"]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
        .rename(
            columns={
                "mean": "ref_mean",
                "std": "ref_std",
                "min": "ref_min",
                "max": "ref_max",
            }
        )
    )

    return prox_summary, ref_summary


def compare_lp_irls_refs(refs):
    """
    Compare LP and IRLS reference losses seed-by-seed.
    """
    wide = (
        refs
        .pivot_table(
            index=["dataset", "seed", "split"],
            columns="method",
            values="pinball_loss",
        )
        .reset_index()
    )

    wide["abs_diff_LP_IRLS"] = np.abs(wide["LP"] - wide["IRLS"])

    summary = (
        wide
        .groupby(["dataset", "split"])["abs_diff_LP_IRLS"]
        .agg(["mean", "std", "max"])
        .reset_index()
        .rename(
            columns={
                "mean": "mean_abs_diff",
                "std": "std_abs_diff",
                "max": "max_abs_diff",
            }
        )
    )

    return summary

## CPS

In [ ]:
def summarize_cps_scaling(
    df,
    *,
    group_cols=("N_SUB", "method"),
    metric_cols=(
        "fit_seconds",
        "rmse",
        "coverage",
        "mean_width",
        "winkler",
    ),
    digits=4,
):
    """
    Summarize CPS scaling results.
    """
    group_cols = list(group_cols)
    metric_cols = list(metric_cols)

    return summarize_by_mean_sd(
        df,
        group_cols=group_cols,
        metric_cols=metric_cols,
        digits=digits,
    )

# Plotting

## Synthetic Plots

In [ ]:
def plot_metric_vs_lr_by_batch(
    df,
    *,
    metric_col="coverage",
    method_col="method",
    reference_method="lp",
    target_method="proxsgd",
    group_cols=("experiment", "seed"),
    lr_col="base_lr",
    bs_col="batch_size",
    agg="median",
    absolute=True,
    logx=True,
    figsize=(7, 5),
    ylabel=None,
    title=None,
):
    """
    Plot target-vs-reference metric gap against base learning rate,
    with one line per mini-batch size.

    Designed for synthetic robustness outputs from run_synthetic_experiment_table.

    Default:
        |proxSGD coverage - LP coverage|
    """
    required = [
        metric_col,
        method_col,
        lr_col,
        bs_col,
        *group_cols,
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(
            f"Missing columns {missing}. "
            f"Available columns include: {list(df.columns)[:25]} ..."
        )

    data = df.copy()
    data[method_col] = data[method_col].astype(str).str.lower()

    reference_method = reference_method.lower()
    target_method = target_method.lower()

    ref = (
        data[data[method_col] == reference_method]
        .loc[:, list(group_cols) + [metric_col]]
        .drop_duplicates(subset=list(group_cols))
        .rename(columns={metric_col: "reference_metric"})
    )

    target = data[data[method_col] == target_method].copy()

    if ref.empty:
        raise ValueError(f"No rows found for reference_method='{reference_method}'.")

    if target.empty:
        raise ValueError(f"No rows found for target_method='{target_method}'.")

    merged = target.merge(ref, on=list(group_cols), how="left")

    if merged["reference_metric"].isna().all():
        raise ValueError(
            "No matching reference rows found. "
            "Try adjusting group_cols, e.g. group_cols=('experiment', 'seed')."
        )

    merged["metric_gap"] = merged[metric_col] - merged["reference_metric"]

    if absolute:
        merged["metric_gap"] = merged["metric_gap"].abs()

    agg_fn = np.nanmedian if agg == "median" else np.nanmean

    grouped = (
        merged.groupby([bs_col, lr_col], dropna=False)["metric_gap"]
        .agg(agg_fn)
        .reset_index()
    )

    batch_sizes = sorted(grouped[bs_col].dropna().unique())

    plt.figure(figsize=figsize)

    for bs in batch_sizes:
        sub = grouped[grouped[bs_col] == bs].sort_values(lr_col)

        plt.plot(
            sub[lr_col],
            sub["metric_gap"],
            marker="o",
            linewidth=2,
            label=f"batch={int(bs)}",
        )

    if logx:
        plt.xscale("log")

    plt.xlabel(r"Base learning rate $\eta_0$", fontsize=14)

    if ylabel is None:
        ylabel = f"{'Absolute ' if absolute else ''}{metric_col} gap"

    plt.ylabel(ylabel, fontsize=16)

    if title is not None:
        plt.title(title, fontsize=16)

    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

def plot_optimizer_lossgap(
    df,
    *,
    metric_col="mean_test_pinball",
    method_col="method",
    reference_method="lp",
    target_method="proxsgd",
    group_cols=("experiment", "seed"),
    lr_col="base_lr",
    bs_col="batch_size",
    agg="median",
    figsize=(6, 5),
    cmap="viridis",
    percentile_clip=95,
):
    """
    Heatmap of absolute loss gap for target_method vs reference_method.

    Designed for outputs from run_synthetic_experiment_table(..., mode="interval").

    Computes:
        |target metric - reference metric|

    Default:
        |proxSGD mean_test_pinball - LP mean_test_pinball|
    """
    required = [
        metric_col,
        method_col,
        lr_col,
        bs_col,
        *group_cols,
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(
            f"Missing columns {missing}. "
            f"Available columns include: {list(df.columns)[:25]} ..."
        )

    data = df.copy()
    data[method_col] = data[method_col].str.lower()

    reference_method = reference_method.lower()
    target_method = target_method.lower()

    ref = (
        data[data[method_col] == reference_method]
        .loc[:, list(group_cols) + [metric_col]]
        .rename(columns={metric_col: "reference_metric"})
    )

    target = data[data[method_col] == target_method].copy()

    if ref.empty:
        raise ValueError(f"No rows found for reference_method='{reference_method}'.")

    if target.empty:
        raise ValueError(f"No rows found for target_method='{target_method}'.")

    merged = target.merge(ref, on=list(group_cols), how="left")

    if merged["reference_metric"].isna().all():
        raise ValueError(
            "No matching reference rows found. "
            "Try adjusting group_cols, e.g. group_cols=('experiment', 'seed')."
        )

    merged["abs_loss_gap"] = (
        merged[metric_col] - merged["reference_metric"]
    ).abs()

    agg_fn = np.nanmedian if agg == "median" else np.nanmean

    grouped = (
        merged.groupby([bs_col, lr_col], dropna=False)["abs_loss_gap"]
        .agg(agg_fn)
        .reset_index()
    )

    batch_sizes = np.sort(grouped[bs_col].dropna().unique())
    lrs = np.sort(grouped[lr_col].dropna().unique())

    loss_mat = (
        grouped.pivot(index=bs_col, columns=lr_col, values="abs_loss_gap")
        .loc[batch_sizes, lrs]
        .values
    )

    loss_vmin = np.nanmin(loss_mat)
    loss_vmax = np.nanpercentile(loss_mat, percentile_clip)

    plt.figure(figsize=figsize)

    im = plt.imshow(
        loss_mat,
        aspect="auto",
        origin="lower",
        cmap=cmap,
        vmin=loss_vmin,
        vmax=loss_vmax,
    )

    plt.xlabel(r"Base learning rate $\eta_0$", fontsize=16)
    plt.ylabel(r"Batch size $m$", fontsize=16)

    plt.xticks(range(len(lrs)))
    xticklabels = []
    for lr in lrs:
        exp = np.log10(lr)
        if lr > 0 and np.isclose(exp, round(exp)):
            xticklabels.append(rf"$10^{{{int(round(exp))}}}$")
        else:
            xticklabels.append(f"{lr:g}")
    plt.gca().set_xticklabels(xticklabels)

    plt.yticks(range(len(batch_sizes)), batch_sizes)

    cbar = plt.colorbar(im)
    cbar.set_label("Absolute loss gap", fontsize=16)

    plt.tight_layout()

## Convergence plot

In [ ]:
import matplotlib.ticker as mticker

In [ ]:
def collect_convergence_results_with_references(
    dataset_names,
    seeds,
    tau,
    estimator_cls,
    sgd_init_kwargs,
    sgd_fit_kwargs=None,
    lp_kwargs=None,
    irls_fit_kwargs=None,
    test_size=0.2,
    scale=True,
):
    sgd_fit_kwargs = {} if sgd_fit_kwargs is None else dict(sgd_fit_kwargs)
    lp_kwargs = {} if lp_kwargs is None else dict(lp_kwargs)
    irls_fit_kwargs = {} if irls_fit_kwargs is None else dict(irls_fit_kwargs)

    curve_rows = []
    ref_rows = []

    for dataset_name in dataset_names:
        for seed in seeds:
            split = prepare_benchmark_split(
                dataset_name,
                test_size=test_size,
                random_state=seed,
                scale=scale,
            )

            X_train_fit = split["X_train"]
            X_test_fit = split["X_test"]
            y_train = split["y_train"]
            y_test = split["y_test"]

            # -------------------------
            # LP reference
            # -------------------------
            lp_model = QuantileRegressor(
                quantile=tau,
                **lp_kwargs,
            )

            t0 = time.perf_counter()
            lp_model.fit(X_train_fit, y_train)
            lp_time = time.perf_counter() - t0

            lp_train_pred = lp_model.predict(X_train_fit)
            lp_test_pred = lp_model.predict(X_test_fit)

            ref_rows.extend(
                [
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "LP",
                        "split": "train",
                        "pinball_loss": pinball_loss(y_train, lp_train_pred, tau),
                        "fit_time_sec": lp_time,
                    },
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "LP",
                        "split": "test",
                        "pinball_loss": pinball_loss(y_test, lp_test_pred, tau),
                        "fit_time_sec": lp_time,
                    },
                ]
            )

            # -------------------------
            # IRLS reference
            # -------------------------
            X_train_sm = sm.add_constant(X_train_fit, has_constant="add")
            X_test_sm = sm.add_constant(X_test_fit, has_constant="add")

            irls_model = sm.QuantReg(y_train, X_train_sm)

            t0 = time.perf_counter()
            irls_res = irls_model.fit(q=tau, **irls_fit_kwargs)
            irls_time = time.perf_counter() - t0

            irls_train_pred = np.asarray(
                irls_res.predict(X_train_sm),
                dtype=float,
            )
            irls_test_pred = np.asarray(
                irls_res.predict(X_test_sm),
                dtype=float,
            )

            ref_rows.extend(
                [
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "IRLS",
                        "split": "train",
                        "pinball_loss": pinball_loss(y_train, irls_train_pred, tau),
                        "fit_time_sec": irls_time,
                    },
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "IRLS",
                        "split": "test",
                        "pinball_loss": pinball_loss(y_test, irls_test_pred, tau),
                        "fit_time_sec": irls_time,
                    },
                ]
            )

            # -------------------------
            # proxSGD convergence curve
            # -------------------------
            sgd_kwargs_seeded = dict(sgd_init_kwargs)
            sgd_kwargs_seeded["random_state"] = seed

            prox_model = estimator_cls(
                quantile=tau,
                **sgd_kwargs_seeded,
            )

            prox_model.fit(
                X_train_fit,
                y_train,
                X_monitor=X_test_fit,
                y_monitor=y_test,
                monitor_name="test",
                **sgd_fit_kwargs,
            )

            hist = prox_model.history_

            for it, train_loss, test_loss in zip(
                hist["iter"],
                hist["train_pinball_loss"],
                hist["monitor_pinball_loss"],
            ):
                curve_rows.append(
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "proxSGD",
                        "iteration": it,
                        "split": "train",
                        "pinball_loss": train_loss,
                    }
                )

                curve_rows.append(
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "proxSGD",
                        "iteration": it,
                        "split": "test",
                        "pinball_loss": test_loss,
                    }
                )

    curves = pd.DataFrame(curve_rows)
    references = pd.DataFrame(ref_rows)

    return curves, references

In [ ]:
def plot_convergence_with_references(
    curves,
    refs,
    dataset_names,
    dataset_labels=None,
    figsize_per_panel=(6, 5),
    use_log_scale=True,
):
    """
    Plot proxSGD train/test convergence curves with LP reference losses.

    Shaded bands show mean +/- one standard deviation across seeds.
    Horizontal black lines show mean LP train/test losses across seeds.
    """
    if dataset_labels is None:
        dataset_labels = {
            name: name.replace("_", " ").title()
            for name in dataset_names
        }

    fig, axes = plt.subplots(
        1,
        len(dataset_names),
        figsize=(figsize_per_panel[0] * len(dataset_names), figsize_per_panel[1]),
        sharex=False,
        sharey=False,
    )

    if len(dataset_names) == 1:
        axes = [axes]

    for ax, dataset in zip(axes, dataset_names):

        # -------------------------
        # proxSGD curves
        # -------------------------
        df = curves[
            (curves["dataset"] == dataset)
            & (curves["method"] == "proxSGD")
        ]

        train_df = (
            df[df["split"] == "train"]
            .pivot(index="seed", columns="iteration", values="pinball_loss")
            .sort_index(axis=1)
        )

        test_df = (
            df[df["split"] == "test"]
            .pivot(index="seed", columns="iteration", values="pinball_loss")
            .sort_index(axis=1)
        )

        iters = train_df.columns.to_numpy()

        train_mean = train_df.mean(axis=0).to_numpy()
        train_std = train_df.std(axis=0).to_numpy()

        test_mean = test_df.mean(axis=0).to_numpy()
        test_std = test_df.std(axis=0).to_numpy()

        ax.plot(
            iters,
            train_mean,
            label="proxSGD train",
            linewidth=2,
        )

        ax.plot(
            iters,
            test_mean,
            linestyle="--",
            label="proxSGD test",
            linewidth=2,
        )

        ax.fill_between(
            iters,
            train_mean - train_std,
            train_mean + train_std,
            alpha=0.2,
        )

        ax.fill_between(
            iters,
            test_mean - test_std,
            test_mean + test_std,
            alpha=0.2,
        )

        # -------------------------
        # LP references
        # -------------------------
        lp_df = refs[
            (refs["dataset"] == dataset)
            & (refs["method"] == "LP")
        ]

        lp_train_mean = (
            lp_df[lp_df["split"] == "train"]["pinball_loss"]
            .mean()
        )

        lp_test_mean = (
            lp_df[lp_df["split"] == "test"]["pinball_loss"]
            .mean()
        )

        ax.axhline(
            lp_train_mean,
            linestyle=":",
            linewidth=2,
            color="black",
            label="LP train",
        )

        ax.axhline(
            lp_test_mean,
            linestyle="-.",
            linewidth=2,
            color="black",
            label="LP test",
        )

        # -------------------------
        # Formatting
        # -------------------------
        ax.set_title(dataset_labels.get(dataset, dataset), fontsize=14)

        if use_log_scale:
            ax.set_yscale("log")

        if dataset == "abalone":
            ticks = [0.6, 0.4, 0.3, 0.25]
            ax.set_yticks(ticks)
            ax.set_yticklabels([str(t) for t in ticks])
            ax.yaxis.set_minor_locator(mticker.NullLocator())

        if dataset == "concrete":
            ticks = [2.5, 2.0, 1.7]
            ax.set_yticks(ticks)
            ax.set_yticklabels([str(t) for t in ticks])
            ax.yaxis.set_minor_locator(mticker.NullLocator())

        ax.set_xlabel("Iteration", fontsize=14)
        ax.grid(True, which="both", linewidth=0.5)

    axes[0].set_ylabel("Pinball loss", fontsize=14)
    axes[0].legend(fontsize=12)

    plt.tight_layout()

    return fig, axes

## CPS plots

In [ ]:
def plot_cps_runtime_scaling(
    df,
    *,
    metric_col="metric",
    value_col="value",
    runtime_metric="fit_seconds",
    n_col="N_SUB",
    method_col="method",
    method_label_map=None,
    pretty_methods=("LP", "IRLS", "ProxSGD"),
    figsize=(8.8, 5.8),
    save_path=None,
    dpi=600,
    return_summaries=False,
):
    """
    Recreate the CPS runtime scaling plot used in the paper.

    Assumes a long-format CPS results DataFrame with columns:
      - method
      - N_SUB
      - metric
      - value

    Runtime rows are selected using metric == 'fit_seconds'. Runtime is
    aggregated across quantile fits at each method/sample-size pair using
    the mean, with min/max shown as error bars.
    """
    if method_label_map is None:
        method_label_map = {
            "sklearn_LP": "LP",
            "lp": "LP",
            "LP": "LP",
            "IRLS": "IRLS",
            "irls": "IRLS",
            "proxSGD": "ProxSGD",
            "proxsgd": "ProxSGD",
            "ProxSGD": "ProxSGD",
        }

    color_map = {
        "LP": "C0",
        "IRLS": "C2",
        "ProxSGD": "C1",
    }

    marker_map = {
        "LP": "o",
        "IRLS": "^",
        "ProxSGD": "s",
    }

    # -------------------------------------------------
    # 1) Filter runtime rows
    # -------------------------------------------------
    runtime_df = df[df[metric_col] == runtime_metric].copy()

    runtime_df[value_col] = pd.to_numeric(runtime_df[value_col], errors="coerce")
    runtime_df[n_col] = pd.to_numeric(runtime_df[n_col], errors="coerce")

    runtime_df["method_pretty"] = runtime_df[method_col].map(method_label_map)

    runtime_df = runtime_df.dropna(
        subset=[value_col, n_col, "method_pretty"]
    )

    # -------------------------------------------------
    # 2) Aggregate across quantiles at each sample size
    # -------------------------------------------------
    runtime_summary = (
        runtime_df
        .groupby([method_col, "method_pretty", n_col], as_index=False)[value_col]
        .agg(
            runtime_mean="mean",
            runtime_min="min",
            runtime_max="max",
            n_quantiles="count",
        )
        .sort_values([method_col, n_col])
    )

    # -------------------------------------------------
    # 3) Log-log fits on aggregated means
    # -------------------------------------------------
    fit_info = {}

    for method, sub in runtime_summary.groupby(method_col):
        sub = sub.sort_values(n_col).copy()

        x_raw = sub[n_col].to_numpy(dtype=float)
        y_raw = sub["runtime_mean"].to_numpy(dtype=float)

        mask = (x_raw > 0) & (y_raw > 0)
        if mask.sum() < 2:
            continue

        x = np.log(x_raw[mask])
        y = np.log(y_raw[mask])

        slope, intercept, r_value, p_value, std_err = linregress(x, y)

        sub["fit_hat"] = np.nan
        sub.loc[mask, "fit_hat"] = np.exp(intercept) * x_raw[mask] ** slope

        pretty = sub["method_pretty"].iloc[0]

        fit_info[method] = {
            "pretty": pretty,
            "slope": slope,
            "intercept": intercept,
            "r2": r_value**2,
            "p_value": p_value,
            "std_err": std_err,
            "df": sub,
        }

    slope_summary = (
        pd.DataFrame(
            [
                {
                    "method": method,
                    "pretty": info["pretty"],
                    "loglog_slope": info["slope"],
                    "r_squared": info["r2"],
                    "p_value": info["p_value"],
                    "slope_std_err": info["std_err"],
                    "n_points": len(info["df"]),
                }
                for method, info in fit_info.items()
            ]
        )
        .sort_values("pretty")
        .reset_index(drop=True)
    )

    # -------------------------------------------------
    # 4) Plot
    # -------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)

    for pretty in pretty_methods:
        sub = runtime_summary[
            runtime_summary["method_pretty"] == pretty
        ].sort_values(n_col)

        if sub.empty:
            continue

        y = sub["runtime_mean"].to_numpy()
        yerr_lower = y - sub["runtime_min"].to_numpy()
        yerr_upper = sub["runtime_max"].to_numpy() - y

        ax.errorbar(
            sub[n_col],
            y,
            yerr=[yerr_lower, yerr_upper],
            fmt=marker_map[pretty] + "-",
            linewidth=2.2,
            markersize=6.5,
            capsize=4,
            color=color_map[pretty],
            zorder=3,
        )

    # Dashed log-log fits
    for method, info in fit_info.items():
        sub = info["df"].sort_values(n_col)
        pretty = info["pretty"]

        ax.plot(
            sub[n_col],
            sub["fit_hat"],
            linestyle="--",
            linewidth=2.0,
            color=color_map[pretty],
            alpha=0.9,
            zorder=2,
        )

    # -------------------------------------------------
    # 5) Formatting
    # -------------------------------------------------
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel("Training Sample Size", fontsize=15)
    ax.set_ylabel("Runtime (seconds)", fontsize=15)
    ax.tick_params(axis="both", labelsize=12)

    ax.xaxis.set_major_locator(LogLocator(base=10))
    ax.yaxis.set_major_locator(LogLocator(base=10))
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.yaxis.set_major_formatter(LogFormatterMathtext())

    ax.grid(True, which="major", alpha=0.25)
    ax.grid(False, which="minor")

    method_handles = [
        Line2D(
            [0], [0],
            color=color_map["LP"],
            marker=marker_map["LP"],
            linestyle="-",
            linewidth=2.2,
            markersize=6.5,
            label="LP",
        ),
        Line2D(
            [0], [0],
            color=color_map["IRLS"],
            marker=marker_map["IRLS"],
            linestyle="-",
            linewidth=2.2,
            markersize=6.5,
            label="IRLS",
        ),
        Line2D(
            [0], [0],
            color=color_map["ProxSGD"],
            marker=marker_map["ProxSGD"],
            linestyle="-",
            linewidth=2.2,
            markersize=6.5,
            label="ProxSGD",
        ),
    ]

    style_handles = [
        Line2D(
            [0], [0],
            color="black",
            linestyle="-",
            marker="o",
            linewidth=2.2,
            markersize=6,
            label="Mean across quantiles",
        ),
        Line2D(
            [0], [0],
            color="black",
            linestyle="None",
            marker="_",
            markersize=12,
            label="Min-max across quantiles",
        ),
        Line2D(
            [0], [0],
            color="black",
            linestyle="--",
            linewidth=2.0,
            label="Log-log fit",
        ),
    ]

    legend1 = ax.legend(
        handles=method_handles,
        loc="upper left",
        frameon=True,
        fontsize=11.5,
        title="Method",
        title_fontsize=12,
    )
    ax.add_artist(legend1)

    ax.legend(
        handles=style_handles,
        loc=(0.4, 0.05),
        frameon=True,
        fontsize=11.0,
        title_fontsize=12,
    )

    annotation_lines = []

    # Match paper ordering where available
    method_order = ["sklearn_LP", "lp", "LP", "IRLS", "irls", "proxSGD", "proxsgd", "ProxSGD"]
    seen_pretty = set()

    for method in method_order:
        if method in fit_info:
            pretty = fit_info[method]["pretty"]
            if pretty in seen_pretty:
                continue
            slope = fit_info[method]["slope"]
            annotation_lines.append(f"{pretty}: slope ≈ {slope:.2f}")
            seen_pretty.add(pretty)

    ax.text(
        0.98,
        0.10,
        "\n".join(annotation_lines),
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=11,
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            alpha=0.9,
            edgecolor="0.7",
        ),
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")

    if return_summaries:
        return fig, ax, runtime_summary, slope_summary
    else:
        return fig, ax

In [ ]:
def plot_cps_coverage_scaling(
    df,
    *,
    n_col="N_SUB",
    metric_col="metric",
    value_col="value",
    method_col="method",
    method_styles=None,
    agg="mean",
    figsize=(8, 5.5),
    title=None,
    nominal=None,
):
    """
    Plot empirical coverage versus training sample size for CPS scaling results.

    Assumes a long-format DataFrame with columns:
      - method
      - N_SUB
      - metric
      - value
    """
    if method_styles is None:
        method_styles = {
            "sklearn_LP": {"label": "LP", "color": "C0", "linestyle": "-"},
            "lp": {"label": "LP", "color": "C0", "linestyle": "-"},
            "IRLS": {"label": "IRLS", "color": "C2", "linestyle": "-"},
            "irls": {"label": "IRLS", "color": "C2", "linestyle": "-"},
            "proxSGD": {"label": "proxSGD", "color": "C1", "linestyle": "-"},
            "proxsgd": {"label": "proxSGD", "color": "C1", "linestyle": "-"},
        }

    if agg == "mean":
        agg_fn = "mean"
    elif agg == "median":
        agg_fn = "median"
    else:
        agg_fn = agg

    coverage = df[df[metric_col] == "coverage"].copy()

    coverage[value_col] = pd.to_numeric(coverage[value_col], errors="coerce")
    coverage[n_col] = pd.to_numeric(coverage[n_col], errors="coerce")

    coverage = coverage.dropna(subset=[method_col, n_col, value_col])

    coverage_tbl = (
        coverage
        .groupby([method_col, n_col], as_index=False)[value_col]
        .agg(agg_fn)
        .sort_values([method_col, n_col])
    )

    fig, ax = plt.subplots(figsize=figsize)

    for method, style in method_styles.items():
        if method not in coverage_tbl[method_col].unique():
            continue

        sub = coverage_tbl[
            coverage_tbl[method_col] == method
        ].sort_values(n_col)

        ax.plot(
            sub[n_col],
            sub[value_col],
            marker="o",
            linewidth=2,
            color=style.get("color", None),
            linestyle=style.get("linestyle", "-"),
            label=style.get("label", str(method)),
        )

    ax.set_xscale("log")
    ax.set_xlabel("Training Sample Size", fontsize=15)
    ax.set_ylabel("Empirical coverage", fontsize=15)

    if nominal is not None:
        ax.axhline(y=nominal, color='black', linestyle='--', linewidth=1)

    if title is not None:
        ax.set_title(title, fontsize=15)

    ax.tick_params(axis="both", labelsize=12)
    ax.legend(fontsize=12)
    ax.grid(True)
    plt.tight_layout()

    return fig, ax

In [ ]:
def plot_cps_width_scaling(
    df,
    *,
    n_col="N_SUB",
    metric_col="metric",
    value_col="value",
    method_col="method",
    method_styles=None,
    agg="mean",
    figsize=(8, 5.5),
    title=None,
    ylabel="Mean interval length",
):
    """
    Plot mean interval width/length versus training sample size for CPS scaling results.

    Assumes a long-format DataFrame with columns:
      - method
      - N_SUB
      - metric
      - value
    """
    if method_styles is None:
        method_styles = {
            "sklearn_LP": {"label": "LP", "color": "C0", "linestyle": "-"},
            "lp": {"label": "LP", "color": "C0", "linestyle": "-"},
            "IRLS": {"label": "IRLS", "color": "C2", "linestyle": "-"},
            "irls": {"label": "IRLS", "color": "C2", "linestyle": "-"},
            "proxSGD": {"label": "proxSGD", "color": "C1", "linestyle": "-"},
            "proxsgd": {"label": "proxSGD", "color": "C1", "linestyle": "-"},
        }

    if agg == "mean":
        agg_fn = "mean"
    elif agg == "median":
        agg_fn = "median"
    else:
        agg_fn = agg

    width = df[df[metric_col] == "mean_width"].copy()

    width[value_col] = pd.to_numeric(width[value_col], errors="coerce")
    width[n_col] = pd.to_numeric(width[n_col], errors="coerce")

    width = width.dropna(subset=[method_col, n_col, value_col])

    width_tbl = (
        width
        .groupby([method_col, n_col], as_index=False)[value_col]
        .agg(agg_fn)
        .sort_values([method_col, n_col])
    )

    fig, ax = plt.subplots(figsize=figsize)

    for method, style in method_styles.items():
        if method not in width_tbl[method_col].unique():
            continue

        sub = width_tbl[
            width_tbl[method_col] == method
        ].sort_values(n_col)

        ax.plot(
            sub[n_col],
            sub[value_col],
            marker="o",
            linewidth=2,
            color=style.get("color", None),
            linestyle=style.get("linestyle", "-"),
            label=style.get("label", str(method)),
        )

    ax.set_xscale("log")
    ax.set_xlabel("Training Sample Size", fontsize=15)
    ax.set_ylabel(ylabel, fontsize=15)

    if title is not None:
        ax.set_title(title, fontsize=15)

    ax.tick_params(axis="both", labelsize=12)
    ax.legend(fontsize=12)
    ax.grid(True)
    plt.tight_layout()

    return fig, ax